In [ ]:
from pathlib import Path
import os
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

print("\nInput folders:")
for p in INPUT.iterdir():
    print(" -", p)

print("\nSearching for competition release roots...")
config_files = list(INPUT.rglob("corridors.json"))

if not config_files:
    print(
        "\nDataset not found yet. Use Add Input to attach "
        "the competition dataset, then rerun this cell."
    )
else:
    for f in config_files:
        root = f.parent
        print("\nCandidate release root:", root)
        print("Contains:")
        for name in [
            "config", "corridors", "task1",
            "task2", "task4", "sample_submission.csv",
            "submission_key.csv"
        ]:
            print(f"  {name}: {(root / name).exists()}")


In [ ]:
# Find the official corridor configuration file
config_files = list(INPUT.rglob("corridors.json"))

print("Number of corridors.json files:", len(config_files))

for config_file in config_files:
    print("\nConfiguration file:")
    print(config_file)

    # The config directory
    config_dir = config_file.parent

    # Its parent directory
    release_root = config_dir.parent

    print("\nConfig directory:", config_dir)
    print("Candidate release root:", release_root)

    print("\nItems inside the candidate release root:")
    for item in sorted(release_root.iterdir()):
        kind = "DIR " if item.is_dir() else "FILE"
        print(f"{kind} | {item.name}")

    print("\nRelevant files found under this root:")
    keywords = (
        "sample_submission",
        "submission_key",
        "train",
        "validation",
        "state",
        "queue",
        "odme",
        "README",
    )

    count = 0
    for item in release_root.rglob("*"):
        if item.is_file() and any(
            word.lower() in item.name.lower()
            for word in keywords
        ):
            print(item.relative_to(release_root))
            count += 1

            if count >= 60:
                print("Stopped after 60 matching files.")
                break


In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/"
    "2026-ieee-big-data-traffic-flow-bench/"
    "kaggle_public"
)

assert RELEASE.exists(), "Release folder not found!"

print("Release folder:", RELEASE)
print("Task 1 exists:", (RELEASE / "task1").exists())
print("Task 2 exists:", (RELEASE / "task2").exists())
print("Task 4 exists:", (RELEASE / "task4").exists())

In [ ]:
import pandas as pd

PANEL = "D12_I5_N"

template_path = (
    RELEASE
    / "task1"
    / PANEL
    / "train"
    / "sample_submission_state.csv"
)

print("Template exists:", template_path.exists())

if template_path.exists():
    template = pd.read_csv(template_path)

    print("Rows and columns:", template.shape)
    print("\nColumn names:")
    print(template.columns.tolist())

    print("\nFirst 5 rows:")
    display(template.head())
else:
    print("Available files in this panel:")
    panel_path = RELEASE / "task1" / PANEL

    for p in panel_path.rglob("*"):
        if p.is_file():
            print(p.relative_to(panel_path))


In [ ]:
TRAIN_DIR = RELEASE / "corridors" / PANEL / "train"

print("Training directory exists:", TRAIN_DIR.exists())

for folder_name in [
    "mainline_states",
    "mainline_states_masked",
    "ramp_flows",
]:
    folder = TRAIN_DIR / folder_name

    print(f"\n{'=' * 50}")
    print("Folder:", folder_name)
    print("Exists:", folder.exists())

    if not folder.exists():
        continue

    parquet_files = sorted(folder.rglob("*.parquet"))

    print("Parquet file count:", len(parquet_files))

    for file in parquet_files[:5]:
        print(" -", file.relative_to(TRAIN_DIR))

    if parquet_files:
        try:
            import pyarrow.parquet as pq

            parquet_file = pq.ParquetFile(parquet_files[0])

            print("\nFirst file schema:")
            print(parquet_file.schema)

            print("Rows in first file:",
                  parquet_file.metadata.num_rows)

        except ImportError:
            print(
                "pyarrow is unavailable. "
                "We will inspect the file with pandas instead."
            )

In [ ]:
# 1. Select the first file from the original data
original_dir = TRAIN_DIR / "mainline_states"
original_files = sorted(original_dir.rglob("*.parquet"))

original_file = original_files[0]

print("Original file:", original_file.name)

# 2. Load the original data
original_df = pd.read_parquet(original_file)

print("\nOriginal data shape:", original_df.shape)
print("\nColumns:")
print(original_df.columns.tolist())

# 3. Display a small sample of the data
display(original_df.head())

# 4. Inspect column data types and missing values
print("\nData types and non-null counts:")
original_df.info()

print("\nMissing values:")
display(
    original_df.isna().sum().sort_values(ascending=False)
)

# 5. Find masked files matching the same day
masked_dir = TRAIN_DIR / "mainline_states_masked"

matching_files = sorted(masked_dir.rglob(original_file.name))

print("\nMatching masked files:", len(matching_files))

for file in matching_files:
    print("\nMasked file:", file.relative_to(TRAIN_DIR))

    masked_df = pd.read_parquet(file)

    print("Shape:", masked_df.shape)
    print(
        "Missing speed values:",
        masked_df["speed_kmh"].isna().sum()
    )
    print(
        "Missing flow values:",
        masked_df["flow_vph"].isna().sum()
    )

# 6. Discover the actual folders inside the training directory
print("\nTop-level folders inside train:")

for path in sorted(TRAIN_DIR.iterdir()):
    if path.is_dir():
        print("-", path.name)


In [ ]:
# 1. Inspect observation status in the original data
print("ORIGINAL DATA")
for column in [
    "is_observed",
    "is_imputed",
    "is_score_eligible",
    "is_missing",
]:
    print(f"\n{column}:")
    print(original_df[column].value_counts(dropna=False).sort_index())

# 2. Inspect the matching R3 masked data
r3_files = [
    file for file in matching_files
    if "mask_regime=R3" in str(file)
]

if r3_files:
    r3_df = pd.read_parquet(r3_files[0])

    print("\nR3 MASKED DATA")
    for column in [
        "is_observed",
        "is_imputed",
        "is_score_eligible",
        "is_missing",
        "mask_regime",
    ]:
        print(f"\n{column}:")
        print(r3_df[column].value_counts(dropna=False).sort_index())

# 3. Locate the submission template
template_path = (
    RELEASE
    / "task1"
    / PANEL
    / "train"
    / "sample_submission_state.csv"
)

print("\nTemplate exists:", template_path.exists())

# 4. Preview the template without loading the entire file
template_preview = pd.read_csv(template_path, nrows=10)

print("\nTEMPLATE COLUMNS:")
print(template_preview.columns.tolist())

print("\nFIRST 10 TEMPLATE ROWS:")
display(template_preview)

# 5. Count template rows and regimes in chunks
total_rows = 0
regime_counts = {}
missing_speed = 0
missing_flow = 0

for chunk in pd.read_csv(template_path, chunksize=500_000):
    total_rows += len(chunk)

    if "mask_regime" in chunk.columns:
        counts = chunk["mask_regime"].value_counts(dropna=False)
        for regime, count in counts.items():
            regime_counts[regime] = (
                regime_counts.get(regime, 0) + int(count)
            )

    missing_speed += int(chunk["speed_kmh"].isna().sum())
    missing_flow += int(chunk["flow_vph"].isna().sum())

print("\nTEMPLATE SUMMARY")
print("Total rows:", total_rows)
print("Rows by mask regime:", regime_counts)
print("Missing speed values:", missing_speed)
print("Missing flow values:", missing_flow)

In [ ]:
# 1. Define the matching keys
KEYS = ["timestamp", "station_id", "link_id"]
TARGET_DATE = "2030-06-01"

# 2. Read only the R3 template rows for the selected date
template_parts = []

for chunk in pd.read_csv(template_path, chunksize=500_000):
    date_mask = chunk["timestamp"].str.startswith(TARGET_DATE)
    regime_mask = chunk["mask_regime"].eq("R3")

    selected = chunk.loc[date_mask & regime_mask].copy()

    if not selected.empty:
        template_parts.append(selected)

template_day = pd.concat(template_parts, ignore_index=True)

print("Template rows for selected date:", len(template_day))

# 3. Check whether the matching keys are unique
print(
    "Duplicate template keys:",
    template_day.duplicated(KEYS).sum()
)

print(
    "Duplicate original keys:",
    original_df.duplicated(KEYS).sum()
)

print(
    "Duplicate masked keys:",
    r3_df.duplicated(KEYS).sum()
)

# 4. Match the template with the original observations
original_check = template_day[KEYS].merge(
    original_df[KEYS + ["speed_kmh", "flow_vph"]],
    on=KEYS,
    how="left",
    indicator=True,
    validate="many_to_one"
)

print("\nORIGINAL DATA MATCH")
print(original_check["_merge"].value_counts())

print(
    "Matched rows with known speed:",
    original_check["speed_kmh"].notna().sum()
)

print(
    "Matched rows with known flow:",
    original_check["flow_vph"].notna().sum()
)

# 5. Match the template with the R3 masked observations
masked_check = template_day[KEYS].merge(
    r3_df[KEYS + ["speed_kmh", "flow_vph"]],
    on=KEYS,
    how="left",
    indicator=True,
    validate="many_to_one"
)

print("\nR3 MASKED DATA MATCH")
print(masked_check["_merge"].value_counts())

print(
    "Matched rows with known speed:",
    masked_check["speed_kmh"].notna().sum()
)

print(
    "Matched rows with known flow:",
    masked_check["flow_vph"].notna().sum()
)

In [ ]:
# 1. Compare missing values in the original and R3 data
original_missing_speed = original_df["speed_kmh"].isna().sum()
masked_missing_speed = r3_df["speed_kmh"].isna().sum()

original_missing_flow = original_df["flow_vph"].isna().sum()
masked_missing_flow = r3_df["flow_vph"].isna().sum()

print("Additional missing speed values:", 
      masked_missing_speed - original_missing_speed)

print("Additional missing flow values:", 
      masked_missing_flow - original_missing_flow)

print("R3 template rows:", len(template_day))

# 2. Check whether every target row was known in the original data
original_targets = template_day[KEYS].merge(
    original_df[KEYS + ["speed_kmh", "flow_vph"]],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("\nTarget rows with known original speed:",
      original_targets["speed_kmh"].notna().sum())

print("Target rows with known original flow:",
      original_targets["flow_vph"].notna().sum())

# 3. Check whether all target values are hidden in R3
masked_targets = template_day[KEYS].merge(
    r3_df[KEYS + ["speed_kmh", "flow_vph"]],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("\nTarget rows with missing R3 speed:",
      masked_targets["speed_kmh"].isna().sum())

print("Target rows with missing R3 flow:",
      masked_targets["flow_vph"].isna().sum())

# 4. Verify the expected counts
print("\nAll checks passed:",
      (masked_missing_speed - original_missing_speed == len(template_day))
      and
      (masked_missing_flow - original_missing_flow == len(template_day))
      and
      original_targets["speed_kmh"].notna().all()
      and
      original_targets["flow_vph"].notna().all()
      and
      masked_targets["speed_kmh"].isna().all()
      and
      masked_targets["flow_vph"].isna().all())

In [ ]:
# Define the repository location
REPO = Path("/kaggle/working/trafficflowbench-public")

# Clone the repository only if it does not already exist
if not REPO.exists():
    !git clone https://github.com/jacky850/trafficflowbench-public.git /kaggle/working/trafficflowbench-public

# Verify that the baseline scripts exist
BASELINE_SCRIPT = REPO / "src" / "task1" / "build_task1_baseline_submission.py"
SCORER_SCRIPT = REPO / "src" / "task1" / "score_task1.py"

print("Baseline script exists:", BASELINE_SCRIPT.exists())
print("Scorer script exists:", SCORER_SCRIPT.exists())

In [ ]:
import subprocess

# Define the competition data and output paths
RELEASE_PATH = str(RELEASE)
OUTPUT_PATH = "/kaggle/working/state_submission_D12_I5_N.csv"

# Build the official historical-mean baseline
command = [
    sys.executable,
    str(BASELINE_SCRIPT),
    "--release-root", RELEASE_PATH,
    "--split", "train",
    "--panel", "D12_I5_N",
    "--output", OUTPUT_PATH,
]

result = subprocess.run(
    command,
    cwd=str(REPO),
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Baseline generation failed.")

# Verify the generated submission file
output_file = Path(OUTPUT_PATH)

In [ ]:
# Evaluate the baseline against the released training targets
command = [
    sys.executable,
    str(SCORER_SCRIPT),
    "--submission", OUTPUT_PATH,
    "--release-root", str(RELEASE),
    "--split", "train",
    "--panel", "D12_I5_N",
]

result = subprocess.run(
    command,
    cwd=str(REPO),
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Baseline evaluation failed.")


In [ ]:
# Load the baseline evaluation report
REPORT_PATH = REPO / "reports" / "task1_eval_validation.csv"
report_df = pd.read_csv(REPORT_PATH)

# Display the evaluation report columns
print("Report columns:")
print(report_df.columns.tolist())

# Filter the selected panel's regime-level results
regime_results = report_df[
    (report_df["level"] == "panel_regime")
    & (report_df["panel"] == PANEL)
]

# Display the most relevant metrics
metrics = [
    "panel",
    "regime",
    "n_target_cells",
    "n_missing_predictions",
    "rmse_speed_kmh",
    "rmse_flow_vph_per_lane",
    "S_speed",
    "S_flow",
    "S_state",
]

display(regime_results[metrics])

# Display the overall result for this panel
panel_result = report_df[
    (report_df["level"] == "panel")
    & (report_df["panel"] == PANEL)
]

print("\nPanel-level score:")
display(panel_result[metrics])

In [ ]:
# 1. Prepare timestamps and matching keys
targets = template_day[KEYS].copy()
targets["timestamp"] = pd.to_datetime(targets["timestamp"], utc=True)

observations = r3_df[
    KEYS + ["speed_kmh", "flow_vph"]
].copy()

observations["timestamp"] = pd.to_datetime(
    observations["timestamp"], utc=True
)

# 2. Prepare observations from five minutes before each target
previous = observations.copy()
previous["timestamp"] = previous["timestamp"] + pd.Timedelta(minutes=5)

previous = previous.rename(
    columns={
        "speed_kmh": "speed_prev",
        "flow_vph": "flow_prev",
    }
)

# 3. Prepare observations from five minutes after each target
following = observations.copy()
following["timestamp"] = following["timestamp"] - pd.Timedelta(minutes=5)

following = following.rename(
    columns={
        "speed_kmh": "speed_next",
        "flow_vph": "flow_next",
    }
)

# 4. Match the temporal observations to each target
temporal_check = targets.merge(
    previous,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

temporal_check = temporal_check.merge(
    following,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

# 5. Count available temporal observations
for column in [
    "speed_prev",
    "speed_next",
    "flow_prev",
    "flow_next",
]:
    print(
        f"{column}: "
        f"{temporal_check[column].notna().sum():,} available values"
    )

# 6. Count targets with at least one usable temporal neighbor
has_previous = (
    temporal_check["speed_prev"].notna()
    & temporal_check["flow_prev"].notna()
)

has_following = (
    temporal_check["speed_next"].notna()
    & temporal_check["flow_next"].notna()
)

print("\nTargets with previous observation:", f"{has_previous.sum():,}")
print("Targets with following observation:", f"{has_following.sum():,}")
print(
    "Targets with at least one temporal neighbor:",
    f"{(has_previous | has_following).sum():,}"
)

In [ ]:
# Standardize timestamp columns
for df in [evaluation_df, temporal_check]:
    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True,
        errors="coerce"
    )

# Check timestamp data types
print("evaluation_df:", evaluation_df["timestamp"].dtype)
print("temporal_check:", temporal_check["timestamp"].dtype)

# Check for invalid timestamps
print(
    "Invalid timestamps:",
    evaluation_df["timestamp"].isna().sum(),
    temporal_check["timestamp"].isna().sum()
)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

temporal_columns = [
    "speed_prev",
    "speed_next",
    "flow_prev",
    "flow_next",
]

evaluation_df = evaluation_df.merge(
    temporal_check[KEYS + temporal_columns],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("Evaluation rows:", len(evaluation_df))
print(
    "Targets with temporal data:",
    evaluation_df[
        ["speed_prev", "speed_next", "flow_prev", "flow_next"]
    ].notna().any(axis=1).sum()
)

In [ ]:
# Create temporal predictions from available neighboring observations
evaluation_df["speed_temporal"] = evaluation_df[
    ["speed_prev", "speed_next"]
].mean(axis=1)

evaluation_df["flow_temporal"] = evaluation_df[
    ["flow_prev", "flow_next"]
].mean(axis=1)

# Fall back to baseline predictions when temporal data is unavailable
evaluation_df["speed_hybrid"] = evaluation_df[
    "speed_temporal"
].fillna(evaluation_df["speed_baseline"])

evaluation_df["flow_hybrid"] = evaluation_df[
    "flow_temporal"
].fillna(evaluation_df["flow_baseline"])

# Calculate RMSE using the same valid target rows
def calculate_rmse(actual, predicted):
    valid = actual.notna() & predicted.notna()

    return np.sqrt(
        mean_squared_error(
            actual.loc[valid],
            predicted.loc[valid]
        )
    )

# Compare baseline and hybrid models
results = pd.DataFrame({
    "Model": ["Historical Baseline", "Temporal Hybrid"],
    "Speed RMSE": [
        calculate_rmse(
            evaluation_df["speed_true"],
            evaluation_df["speed_baseline"]
        ),
        calculate_rmse(
            evaluation_df["speed_true"],
            evaluation_df["speed_hybrid"]
        ),
    ],
    "Flow RMSE": [
        calculate_rmse(
            evaluation_df["flow_true"],
            evaluation_df["flow_baseline"]
        ),
        calculate_rmse(
            evaluation_df["flow_true"],
            evaluation_df["flow_hybrid"]
        ),
    ],
})

display(results)

# Calculate percentage improvement over the baseline
baseline_result = results.iloc[0]
hybrid_result = results.iloc[1]

speed_improvement = 100 * (
    baseline_result["Speed RMSE"] - hybrid_result["Speed RMSE"]
) / baseline_result["Speed RMSE"]

flow_improvement = 100 * (
    baseline_result["Flow RMSE"] - hybrid_result["Flow RMSE"]
) / baseline_result["Flow RMSE"]

print(f"Speed RMSE improvement: {speed_improvement:.2f}%")
print(f"Flow RMSE improvement: {flow_improvement:.2f}%")

In [ ]:
# Count available temporal observations for each target
evaluation_df["n_speed_neighbors"] = evaluation_df[
    ["speed_prev", "speed_next"]
].notna().sum(axis=1)

evaluation_df["n_flow_neighbors"] = evaluation_df[
    ["flow_prev", "flow_next"]
].notna().sum(axis=1)

# Summarize temporal coverage
coverage = pd.DataFrame({
    "Speed": evaluation_df["n_speed_neighbors"].value_counts().sort_index(),
    "Flow": evaluation_df["n_flow_neighbors"].value_counts().sort_index(),
}).fillna(0).astype(int)

coverage.index.name = "Available temporal neighbors"
display(coverage)

# Report target coverage percentages
print(
    "Targets with at least one speed neighbor:",
    f"{(evaluation_df['n_speed_neighbors'].gt(0).mean() * 100):.2f}%"
)

print(
    "Targets with both speed neighbors:",
    f"{(evaluation_df['n_speed_neighbors'].eq(2).mean() * 100):.2f}%"
)

In [ ]:
# Locate the competition documentation
readme_path = RELEASE / "README.md"
task1_dir = RELEASE / "task1"

print("README path:", readme_path)
print("README exists:", readme_path.exists())

if readme_path.exists():
    print("\n--- Competition README ---")
    print(readme_path.read_text(errors="replace")[:12000])

# List Task 1 documentation files
print("\n--- Task 1 files ---")
for path in sorted(task1_dir.rglob("*")):
    if path.is_file() and (
        path.suffix.lower() in {".md", ".txt", ".yaml", ".yml", ".json"}
    ):
        print(path)

In [ ]:
import re

# Define the training and submission paths
train_dir = RELEASE / "corridors" / PANEL / "train"
template_path = RELEASE / "task1" / PANEL / "train" / "sample_submission_state.csv"
baseline_path = Path("/kaggle/working/state_submission_D12_I5_N.csv")
hybrid_path = Path("/kaggle/working/state_submission_hybrid_D12_I5_N.csv")

# Load the target template and baseline predictions
targets = pd.read_csv(template_path)
baseline_df = pd.read_csv(baseline_path)

# Standardize join keys
for df in [targets, baseline_df]:
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], utc=True, errors="coerce"
    )
    df["station_id"] = df["station_id"].astype(str)
    df["link_id"] = df["link_id"].astype(str)
    df["mask_regime"] = df["mask_regime"].astype(str)

targets["date_key"] = targets["timestamp"].dt.strftime("%Y_%m_%d")

# Index target rows by regime and date
target_groups = {
    key: group.drop(columns=["date_key"]).copy()
    for key, group in targets.groupby(
        ["mask_regime", "date_key"], sort=False
    )
}

temporal_parts = []
regimes = ["R1", "R2", "R3"]
neighbor_columns = ["timestamp", "station_id", "link_id"]

# Process one daily masked file at a time
for regime in regimes:
    masked_dir = (
        train_dir
        / "mainline_states_masked"
        / f"mask_regime={regime}"
    )

    files = sorted(masked_dir.glob("*.parquet"))
    print(f"{regime}: processing {len(files)} files")

    for file_path in files:
        match = re.search(r"(\d{4}_\d{2}_\d{2})", file_path.stem)
        if match is None:
            continue

        date_key = match.group(1)
        day_targets = target_groups.get((regime, date_key))

        if day_targets is None or day_targets.empty:
            continue

        # Load only the columns required for temporal predictions
        masked = pd.read_parquet(
            file_path,
            columns=[
                "timestamp",
                "station_id",
                "link_id",
                "speed_kmh",
                "flow_vph",
            ],
        )

        masked["timestamp"] = pd.to_datetime(
            masked["timestamp"], utc=True, errors="coerce"
        )
        masked["station_id"] = masked["station_id"].astype(str)
        masked["link_id"] = masked["link_id"].astype(str)

        # Prepare observations from five minutes earlier
        previous = masked[
            neighbor_columns + ["speed_kmh", "flow_vph"]
        ].copy()

        previous["timestamp"] += pd.Timedelta(minutes=5)
        previous = previous.rename(
            columns={
                "speed_kmh": "speed_prev",
                "flow_vph": "flow_prev",
            }
        )

        # Prepare observations from five minutes later
        following = masked[
            neighbor_columns + ["speed_kmh", "flow_vph"]
        ].copy()

        following["timestamp"] -= pd.Timedelta(minutes=5)
        following = following.rename(
            columns={
                "speed_kmh": "speed_next",
                "flow_vph": "flow_next",
            }
        )

        # Match temporal observations to the target rows
        day_result = day_targets[
            ["panel", "timestamp", "station_id", "link_id", "mask_regime"]
        ].merge(
            previous,
            on=neighbor_columns,
            how="left",
            validate="one_to_one",
        )

        day_result = day_result.merge(
            following,
            on=neighbor_columns,
            how="left",
            validate="one_to_one",
        )

        # Average the available temporal observations
        day_result["speed_temporal"] = day_result[
            ["speed_prev", "speed_next"]
        ].mean(axis=1)

        day_result["flow_temporal"] = day_result[
            ["flow_prev", "flow_next"]
        ].mean(axis=1)

        # Retain rows with at least one available temporal observation
        available = (
            day_result["speed_temporal"].notna()
            & day_result["flow_temporal"].notna()
        )

        temporal_parts.append(
            day_result.loc[
                available,
                [
                    "panel",
                    "timestamp",
                    "station_id",
                    "link_id",
                    "mask_regime",
                    "speed_temporal",
                    "flow_temporal",
                ],
            ]
        )

    print(f"{regime}: finished")

# Combine temporal predictions from all daily files
temporal_predictions = pd.concat(
    temporal_parts,
    ignore_index=True,
)

merge_keys = [
    "panel",
    "timestamp",
    "station_id",
    "link_id",
    "mask_regime",
]

# Check temporal prediction uniqueness
duplicate_count = temporal_predictions.duplicated(merge_keys).sum()

print("Temporal prediction rows:", len(temporal_predictions))
print("Duplicate temporal keys:", duplicate_count)

if duplicate_count:
    raise ValueError("Duplicate temporal prediction keys were found.")

# Merge temporal predictions into the baseline
hybrid_df = baseline_df.merge(
    temporal_predictions,
    on=merge_keys,
    how="left",
    validate="one_to_one",
)

# Replace baseline values only where temporal predictions are available
hybrid_df["speed_kmh"] = hybrid_df["speed_temporal"].fillna(
    hybrid_df["speed_kmh"]
)

hybrid_df["flow_vph"] = hybrid_df["flow_temporal"].fillna(
    hybrid_df["flow_vph"]
)

# Preserve the required submission columns and row count
submission_columns = [
    "panel",
    "timestamp",
    "station_id",
    "link_id",
    "mask_regime",
    "speed_kmh",
    "flow_vph",
]

hybrid_df[submission_columns].to_csv(hybrid_path, index=False)

print("Baseline rows:", len(baseline_df))
print("Hybrid rows:", len(hybrid_df))
print("Saved hybrid submission:", hybrid_path)
print(
    "Missing speed predictions:",
    hybrid_df["speed_kmh"].isna().sum(),
)
print(
    "Missing flow predictions:",
    hybrid_df["flow_vph"].isna().sum(),
)


In [ ]:
# Run the official Task 1 evaluator on the hybrid submission
command = [
    sys.executable,
    str(REPO / "src" / "task1" / "score_task1.py"),
    "--submission",
    str(hybrid_path),
    "--release-root",
    str(RELEASE),
    "--split",
    "train",
    "--panel",
    PANEL,
    "--output",
    "/kaggle/working/task1_eval_hybrid.csv",
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
)


In [ ]:
# Inspect the official baseline implementation
baseline_script = REPO / "src" / "task1" / "build_task1_baseline_submission.py"

source = baseline_script.read_text(errors="replace")

print(source)

In [ ]:
# Locate the historical baseline implementation
baseline_module = (
    REPO / "src" / "task1" / "baseline_task1_historical_mean.py"
)

# Display the profile-building implementation
source = baseline_module.read_text(errors="replace")

lines = source.splitlines()

for i, line in enumerate(lines, start=1):
    if (
        "def build_profile" in line
        or "def masked_files" in line
        or "def slot_values" in line
    ):
        start = max(0, i - 5)
        end = min(len(lines), i + 100)

        print(f"\n--- Source lines {start + 1}-{end} ---")
        print("\n".join(lines[start:end]))

In [ ]:
# Read the historical baseline source
baseline_module = (
    REPO / "src" / "task1" / "baseline_task1_historical_mean.py"
)

source_lines = baseline_module.read_text(
    errors="replace"
).splitlines()

# Display the file-selection helper and its surrounding context
for i, line in enumerate(source_lines):
    if line.startswith("def files("):
        start = max(0, i - 10)
        end = min(len(source_lines), i + 35)

        print("\n".join(
            f"{j + 1}: {source_lines[j]}"
            for j in range(start, end)
        ))
        break
else:
    print("The files() function definition was not found.")

In [ ]:
evaluator = REPO / "src" / "task1" / "score_task1.py"

result = subprocess.run(
    [sys.executable, str(evaluator), "--help"],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

In [ ]:
builder = REPO / "src" / "task1" / "build_task1_baseline_submission.py"

result = subprocess.run(
    [sys.executable, str(builder), "--help"],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

In [ ]:
baseline_output = "/kaggle/working/state_baseline_validation_D12_I5_N.csv"

cmd = [
    sys.executable,
    str(REPO / "src" / "task1" / "build_task1_baseline_submission.py"),
    "--release-root", str(RELEASE),
    "--split", "validation",
    "--panel", PANEL,
    "--output", baseline_output,
]

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Baseline generation failed.")

In [ ]:
files_to_inspect = {
    "evaluator": REPO / "src" / "task1" / "score_task1.py",
    "builder": REPO / "src" / "task1" / "build_task1_baseline_submission.py",
}

for label, path in files_to_inspect.items():
    lines = path.read_text(errors="replace").splitlines()

    print(f"\n{'=' * 25} {label.upper()} {'=' * 25}")

    if label == "evaluator":
        start, end = 70, 145
    else:
        start, end = 1, min(180, len(lines))

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")

In [ ]:
train_dir = RELEASE / "corridors" / PANEL / "train"

truth_dir = train_dir / "mainline_states"
masked_dir = train_dir / "mainline_states_masked"

truth_files = sorted(truth_dir.glob("**/*.parquet"))
masked_files = sorted(masked_dir.glob("**/*.parquet"))

print("Original train files:", len(truth_files))
print("Masked train files:", len(masked_files))

truth_names = {p.name for p in truth_files}
masked_names = {p.name for p in masked_files}

print("Matching filenames:", len(truth_names & masked_names))
print("Original-only files:", len(truth_names - masked_names))
print("Masked-only files:", len(masked_names - truth_names))

if truth_files:
    sample = pd.read_parquet(
        truth_files[0],
        columns=["date", "timestamp", "station_id", "link_id",
                 "speed_kmh", "flow_vph", "is_score_eligible"]
    )
    print("\nSample date:", sample["date"].iloc[0])
    print("Sample rows:", len(sample))
    print("Score-eligible rows:", int(sample["is_score_eligible"].astype(bool).sum()))
    print("Sample columns:", sample.columns.tolist())

In [ ]:
train_dir = RELEASE / "corridors" / PANEL / "train"
truth_dir = train_dir / "mainline_states"
masked_dir = train_dir / "mainline_states_masked"

truth_path = sorted(truth_dir.glob("**/*.parquet"))[0]

print("Original file:")
print(truth_path)

print("\nMatching masked files:")
matches = list(masked_dir.glob(f"**/{truth_path.name}"))

for path in matches:
    print(path)

print("\nMasked directory sample:")
for path in list(masked_dir.glob("**/*.parquet"))[:5]:
    print(path)


In [ ]:
truth_path = sorted(truth_dir.glob("**/synthetic_mainline_2030_06_01.parquet"))[0]

masked_matches = list(
    masked_dir.glob("mask_regime=R3/**/synthetic_mainline_2030_06_01.parquet")
)

assert len(masked_matches) == 1, f"Expected one R3 file, found {len(masked_matches)}"

masked_path = masked_matches[0]

truth = pd.read_parquet(truth_path)
masked = pd.read_parquet(masked_path)

print("Original file:", truth_path)
print("Masked file:", masked_path)
print("Truth shape:", truth.shape)
print("Masked shape:", masked.shape)

keys = ["timestamp", "station_id", "link_id"]

print("\nDuplicate keys:")
print("Truth:", int(truth.duplicated(keys).sum()))
print("Masked:", int(masked.duplicated(keys).sum()))

truth_values = truth[keys + ["speed_kmh", "flow_vph"]].copy()
masked_values = masked[keys + ["speed_kmh", "flow_vph"]].copy()

comparison = truth_values.merge(
    masked_values,
    on=keys,
    how="inner",
    suffixes=("_truth", "_masked"),
    validate="one_to_one"
)

print("\nMatched rows:", len(comparison))
print("Original rows:", len(truth))
print("Masked rows:", len(masked))

for col in ["speed_kmh", "flow_vph"]:
    original_valid = comparison[f"{col}_truth"].notna()
    hidden = comparison[f"{col}_masked"].isna()

    print(f"\n{col}")
    print("Original non-missing:", int(original_valid.sum()))
    print("Hidden in masked file:", int((original_valid & hidden).sum()))
    print(
        "Visible in both:",
        int((original_valid & comparison[f"{col}_masked"].notna()).sum())
    )

print("\nMask regime:", "R3")


In [ ]:
template_path = (
    RELEASE / "task1" / PANEL / "train" / "sample_submission_state.csv"
)

template = pd.read_csv(template_path)

day = "2030-06-01"
day_keys = truth[["timestamp", "station_id", "link_id"]].copy()

template_day = template[
    template["timestamp"].astype(str).str.startswith(day)
].copy()

print("Template columns:", template.columns.tolist())
print("Template rows for day:", len(template_day))

target_keys = template_day[["timestamp", "station_id", "link_id"]].drop_duplicates()
print("Unique target keys:", len(target_keys))

truth_day = truth.merge(
    target_keys,
    on=["timestamp", "station_id", "link_id"],
    how="inner",
    validate="one_to_one"
)

print("Truth rows matching template:", len(truth_day))

if "is_score_eligible" in truth_day.columns:
    print(
        "Eligible matching targets:",
        int(truth_day["is_score_eligible"].astype(bool).sum())
    )

print("\nTarget regimes:")
print(template_day["mask_regime"].value_counts(dropna=False))

print("\nMissing targets in template:")
print(
    template_day[["speed_kmh", "flow_vph"]]
    .isna()
    .sum()
)


In [ ]:
import inspect

task1_src = str(REPO / "src" / "task1")
if task1_src not in sys.path:
    sys.path.insert(0, task1_src)

import baseline_task1_historical_mean as historical
import score_task1

print("=" * 70)
print("HISTORICAL MEAN: build_profile")
print("=" * 70)
print(inspect.getsource(historical.build_profile))

print("\n" + "=" * 70)
print("HISTORICAL MEAN: slot_values")
print("=" * 70)
print(inspect.getsource(historical.slot_values))

print("\n" + "=" * 70)
print("SCORING: score-related functions")
print("=" * 70)

for name, obj in inspect.getmembers(score_task1, inspect.isfunction):
    if any(term in name.lower() for term in ["score", "metric", "rmse"]):
        print(f"\n--- {name} ---")
        print(inspect.getsource(obj))


In [ ]:
day = "2030-06-01"
regime = "R3"

train_dir = RELEASE / "corridors" / PANEL / "train"
truth_dir = train_dir / "mainline_states"
masked_dir = train_dir / "mainline_states_masked" / f"mask_regime={regime}"
template_path = RELEASE / "task1" / PANEL / "train" / "sample_submission_state.csv"

truth_path = next(truth_dir.glob(f"**/synthetic_mainline_{day.replace('-', '_')}.parquet"))
masked_path = next(masked_dir.glob(f"**/synthetic_mainline_{day.replace('-', '_')}.parquet"))

truth = pd.read_parquet(truth_path)
masked = pd.read_parquet(masked_path)
template = pd.read_csv(template_path)

keys = ["timestamp", "station_id", "link_id"]
target = template[
    template["timestamp"].astype(str).str.startswith(day)
    & template["mask_regime"].eq(regime)
].copy()

target["timestamp"] = pd.to_datetime(target["timestamp"], utc=True)
target["station_id"] = target["station_id"].astype(str)
target["link_id"] = target["link_id"].astype(str)

truth["timestamp"] = pd.to_datetime(truth["timestamp"], utc=True)
truth["station_id"] = truth["station_id"].astype(str)
truth["link_id"] = truth["link_id"].astype(str)

truth_targets = target[keys].merge(
    truth[keys + ["speed_kmh", "flow_vph", "is_score_eligible"]],
    on=keys,
    how="left",
    validate="one_to_one"
)

assert truth_targets["is_score_eligible"].astype(bool).all()
assert truth_targets[["speed_kmh", "flow_vph"]].notna().all().all()

print("Official targets:", len(target))
print("All targets eligible:", truth_targets["is_score_eligible"].astype(bool).all())

# Build historical profiles while excluding the evaluation day.
first = pd.read_parquet(
    sorted(truth_dir.glob("**/*.parquet"))[0],
    columns=["link_id"]
)
link_ids = sorted(first["link_id"].astype(str).unique())
link_index = {link: i for i, link in enumerate(link_ids)}

n_links = len(link_ids)
n_slots = 7 * 288

speed_sum = np.zeros((n_links, n_slots), dtype=np.float64)
speed_count = np.zeros((n_links, n_slots), dtype=np.int64)
flow_sum = np.zeros((n_links, n_slots), dtype=np.float64)
flow_count = np.zeros((n_links, n_slots), dtype=np.int64)

for i, path in enumerate(sorted(truth_dir.glob("**/*.parquet")), 1):
    if path.name == truth_path.name:
        continue

    frame = pd.read_parquet(
        path,
        columns=["timestamp", "link_id", "speed_kmh", "flow_vph", "pct_observed"]
    )

    frame["link_id"] = frame["link_id"].astype(str)
    li = frame["link_id"].map(link_index).to_numpy(dtype=np.int64)

    ts = pd.to_datetime(frame["timestamp"], utc=True)
    weekday = ts.dt.weekday.to_numpy(dtype=np.int16)
    tod = (ts.dt.hour * 12 + ts.dt.minute // 5).to_numpy(dtype=np.int16)
    slot = weekday * 288 + tod

    speed = pd.to_numeric(frame["speed_kmh"], errors="coerce").to_numpy(dtype=float)
    flow = pd.to_numeric(frame["flow_vph"], errors="coerce").to_numpy(dtype=float)
    pct = pd.to_numeric(frame["pct_observed"], errors="coerce").to_numpy(dtype=float)

    valid = (
        (pct >= 75)
        & np.isfinite(speed)
        & np.isfinite(flow)
    )

    np.add.at(speed_sum, (li[valid], slot[valid]), speed[valid])
    np.add.at(speed_count, (li[valid], slot[valid]), 1)
    np.add.at(flow_sum, (li[valid], slot[valid]), flow[valid])
    np.add.at(flow_count, (li[valid], slot[valid]), 1)

    if i % 50 == 0:
        print(f"Historical profile: {i}/{len(list(truth_dir.glob('**/*.parquet')))} files")

speed_mean = np.divide(
    speed_sum, np.maximum(speed_count, 1),
    out=np.zeros_like(speed_sum), where=speed_count > 0
)
flow_mean = np.divide(
    flow_sum, np.maximum(flow_count, 1),
    out=np.zeros_like(flow_sum), where=flow_count > 0
)

speed_fallback = speed_sum.sum(axis=1) / np.maximum(speed_count.sum(axis=1), 1)
flow_fallback = flow_sum.sum(axis=1) / np.maximum(flow_count.sum(axis=1), 1)

target_ts = target["timestamp"]
weekday = target_ts.dt.weekday.to_numpy(dtype=np.int16)
tod = (target_ts.dt.hour * 12 + target_ts.dt.minute // 5).to_numpy(dtype=np.int16)
slots = weekday * 288 + tod
li = target["link_id"].map(link_index).to_numpy(dtype=np.int64)

base_speed = speed_mean[li, slots].copy()
base_flow = flow_mean[li, slots].copy()

no_speed_slot = speed_count[li, slots] == 0
no_flow_slot = flow_count[li, slots] == 0

base_speed[no_speed_slot] = speed_fallback[li[no_speed_slot]]
base_flow[no_flow_slot] = flow_fallback[li[no_flow_slot]]

# Match temporal neighbors from the masked evaluation day only.
masked["timestamp"] = pd.to_datetime(masked["timestamp"], utc=True)
masked["station_id"] = masked["station_id"].astype(str)
masked["link_id"] = masked["link_id"].astype(str)

visible = masked[
    masked["speed_kmh"].notna() & masked["flow_vph"].notna()
][keys + ["speed_kmh", "flow_vph"]].copy()

previous = visible.copy()
previous["timestamp"] += pd.Timedelta(minutes=5)
previous = previous.rename(columns={
    "speed_kmh": "speed_prev",
    "flow_vph": "flow_prev"
})

following = visible.copy()
following["timestamp"] -= pd.Timedelta(minutes=5)
following = following.rename(columns={
    "speed_kmh": "speed_next",
    "flow_vph": "flow_next"
})

temporal = target[keys].merge(previous, on=keys, how="left", validate="one_to_one")
temporal = temporal.merge(following, on=keys, how="left", validate="one_to_one")

speed_neighbors = temporal[["speed_prev", "speed_next"]].to_numpy(dtype=float)
flow_neighbors = temporal[["flow_prev", "flow_next"]].to_numpy(dtype=float)

speed_n = np.isfinite(speed_neighbors)
flow_n = np.isfinite(flow_neighbors)

speed_count_n = speed_n.sum(axis=1)
flow_count_n = flow_n.sum(axis=1)

temporal_speed = np.divide(
    np.nansum(speed_neighbors, axis=1),
    np.maximum(speed_count_n, 1)
)
temporal_flow = np.divide(
    np.nansum(flow_neighbors, axis=1),
    np.maximum(flow_count_n, 1)
)

hybrid_speed = np.where(speed_count_n > 0, temporal_speed, base_speed)
hybrid_flow = np.where(flow_count_n > 0, temporal_flow, base_flow)

# Calculate RMSE on exactly the same official target rows.
true_speed = truth_targets["speed_kmh"].to_numpy(dtype=float)
true_flow = truth_targets["flow_vph"].to_numpy(dtype=float)

# Resolve lane counts using the competition's scoring utilities.
import sys
task1_src = str(REPO / "src" / "task1")
if task1_src not in sys.path:
    sys.path.insert(0, task1_src)

import score_task1

lanes_by_station = score_task1.station_lanes(RELEASE / "corridors" / PANEL)
lanes = score_task1.lane_vector(target["station_id"], target["link_id"], lanes_by_station)

def metrics(name, pred_speed, pred_flow):
    rmse_speed = float(np.sqrt(np.mean((pred_speed - true_speed) ** 2)))
    rmse_flow = float(np.sqrt(np.mean(((pred_flow - true_flow) / lanes) ** 2)))
    s_speed = max(0.0, 1.0 - rmse_speed / score_task1.SPEED_NORMALIZER)
    s_flow = max(0.0, 1.0 - rmse_flow / score_task1.FLOW_NORMALIZER)
    s_state = score_task1.SPEED_WEIGHT * s_speed + score_task1.FLOW_WEIGHT * s_flow
    return {
        "model": name,
        "RMSE_speed": rmse_speed,
        "RMSE_flow_per_lane": rmse_flow,
        "S_speed": s_speed,
        "S_flow": s_flow,
        "S_state": s_state
    }

results = pd.DataFrame([
    metrics("Historical Mean (day excluded)", base_speed, base_flow),
    metrics("Temporal Hybrid (day excluded)", hybrid_speed, hybrid_flow)
])

print("\nTemporal neighbor coverage:")
print(pd.Series(speed_count_n).value_counts().sort_index().rename("target_count"))
print("\nLeakage-free results:")
display(results)


In [ ]:
from collections import defaultdict
import score_task1

train_dir = RELEASE / "corridors" / PANEL / "train"
truth_dir = train_dir / "mainline_states"
masked_root = train_dir / "mainline_states_masked"
template_path = RELEASE / "task1" / PANEL / "train" / "sample_submission_state.csv"

truth_files = sorted(truth_dir.glob("**/*.parquet"))
template = pd.read_csv(template_path)

KEYS = ["timestamp", "station_id", "link_id"]
N_SLOTS = 7 * 288

first = pd.read_parquet(truth_files[0], columns=["link_id"])
link_ids = sorted(first["link_id"].astype(str).unique())
link_index = {v: i for i, v in enumerate(link_ids)}
n_links = len(link_ids)

def prepare_frame(path):
    frame = pd.read_parquet(path)
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)
    frame["station_id"] = frame["station_id"].astype(str)
    frame["link_id"] = frame["link_id"].astype(str)
    return frame

def profile_contribution(path):
    frame = pd.read_parquet(
        path,
        columns=[
            "timestamp", "link_id", "speed_kmh",
            "flow_vph", "pct_observed"
        ]
    )

    frame["link_id"] = frame["link_id"].astype(str)
    li = frame["link_id"].map(link_index).to_numpy(dtype=np.int64)

    ts = pd.to_datetime(frame["timestamp"], utc=True)
    weekday = ts.dt.weekday.to_numpy(dtype=np.int16)
    tod = (ts.dt.hour * 12 + ts.dt.minute // 5).to_numpy(dtype=np.int16)
    slot = weekday * 288 + tod

    speed = pd.to_numeric(frame["speed_kmh"], errors="coerce").to_numpy(dtype=float)
    flow = pd.to_numeric(frame["flow_vph"], errors="coerce").to_numpy(dtype=float)
    pct = pd.to_numeric(frame["pct_observed"], errors="coerce").to_numpy(dtype=float)

    valid = (
        (pct >= 75)
        & np.isfinite(speed)
        & np.isfinite(flow)
    )

    ss = np.zeros((n_links, N_SLOTS), dtype=np.float64)
    sc = np.zeros((n_links, N_SLOTS), dtype=np.int64)
    fs = np.zeros((n_links, N_SLOTS), dtype=np.float64)
    fc = np.zeros((n_links, N_SLOTS), dtype=np.int64)

    np.add.at(ss, (li[valid], slot[valid]), speed[valid])
    np.add.at(sc, (li[valid], slot[valid]), 1)
    np.add.at(fs, (li[valid], slot[valid]), flow[valid])
    np.add.at(fc, (li[valid], slot[valid]), 1)

    return ss, sc, fs, fc

# Aggregate historical profiles once.
total_ss = np.zeros((n_links, N_SLOTS), dtype=np.float64)
total_sc = np.zeros((n_links, N_SLOTS), dtype=np.int64)
total_fs = np.zeros((n_links, N_SLOTS), dtype=np.float64)
total_fc = np.zeros((n_links, N_SLOTS), dtype=np.int64)

for i, path in enumerate(truth_files, 1):
    ss, sc, fs, fc = profile_contribution(path)
    total_ss += ss
    total_sc += sc
    total_fs += fs
    total_fc += fc

    if i % 50 == 0:
        print(f"Profile aggregation: {i}/{len(truth_files)}")

lanes_by_station = score_task1.station_lanes(
    RELEASE / "corridors" / PANEL
)

squared_errors = defaultdict(lambda: {
    "speed": 0.0,
    "flow": 0.0,
    "n": 0
})

processed_days = 0
skipped_days = []

for day_idx, truth_path in enumerate(truth_files, 1):
    day = truth_path.stem.replace("synthetic_mainline_", "")
    day = day.replace("_", "-")

    day_template = template[
        template["timestamp"].astype(str).str.startswith(day)
    ].copy()

    if day_template.empty:
        continue

    regimes = day_template["mask_regime"].dropna().unique()
    if len(regimes) != 1:
        skipped_days.append((day, "Expected exactly one regime"))
        continue

    regime = regimes[0]
    masked_matches = list(
        (masked_root / f"mask_regime={regime}").glob(
            f"**/{truth_path.name}"
        )
    )

    if len(masked_matches) != 1:
        skipped_days.append((day, "Masked file not found uniquely"))
        continue

    # Exclude this entire day from the historical profile.
    day_ss, day_sc, day_fs, day_fc = profile_contribution(truth_path)

    ss = total_ss - day_ss
    sc = total_sc - day_sc
    fs = total_fs - day_fs
    fc = total_fc - day_fc

    speed_mean = np.divide(
        ss, np.maximum(sc, 1),
        out=np.zeros_like(ss), where=sc > 0
    )
    flow_mean = np.divide(
        fs, np.maximum(fc, 1),
        out=np.zeros_like(fs), where=fc > 0
    )

    speed_fallback = ss.sum(axis=1) / np.maximum(sc.sum(axis=1), 1)
    flow_fallback = fs.sum(axis=1) / np.maximum(fc.sum(axis=1), 1)

    target = day_template.copy()
    target["timestamp"] = pd.to_datetime(target["timestamp"], utc=True)
    target["station_id"] = target["station_id"].astype(str)
    target["link_id"] = target["link_id"].astype(str)

    truth = prepare_frame(truth_path)
    actual = target[KEYS].merge(
        truth[KEYS + ["speed_kmh", "flow_vph", "is_score_eligible"]],
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

    actual = actual[
        actual["is_score_eligible"].astype(bool)
        & actual["speed_kmh"].notna()
        & actual["flow_vph"].notna()
    ].copy()

    if actual.empty:
        continue

    target = actual[KEYS].copy()
    ts = target["timestamp"]
    weekday = ts.dt.weekday.to_numpy(dtype=np.int16)
    tod = (ts.dt.hour * 12 + ts.dt.minute // 5).to_numpy(dtype=np.int16)
    slots = weekday * 288 + tod
    li = target["link_id"].map(link_index).to_numpy(dtype=np.int64)

    base_speed = speed_mean[li, slots].copy()
    base_flow = flow_mean[li, slots].copy()

    no_speed = sc[li, slots] == 0
    no_flow = fc[li, slots] == 0
    base_speed[no_speed] = speed_fallback[li[no_speed]]
    base_flow[no_flow] = flow_fallback[li[no_flow]]

    masked = prepare_frame(masked_matches[0])
    visible = masked[
        masked["speed_kmh"].notna() & masked["flow_vph"].notna()
    ][KEYS + ["speed_kmh", "flow_vph"]].copy()

    previous = visible.copy()
    previous["timestamp"] += pd.Timedelta(minutes=5)
    previous = previous.rename(columns={
        "speed_kmh": "speed_prev",
        "flow_vph": "flow_prev"
    })

    following = visible.copy()
    following["timestamp"] -= pd.Timedelta(minutes=5)
    following = following.rename(columns={
        "speed_kmh": "speed_next",
        "flow_vph": "flow_next"
    })

    temporal = target.merge(
        previous, on=KEYS, how="left", validate="one_to_one"
    ).merge(
        following, on=KEYS, how="left", validate="one_to_one"
    )

    sp_neighbors = temporal[["speed_prev", "speed_next"]].to_numpy(dtype=float)
    fl_neighbors = temporal[["flow_prev", "flow_next"]].to_numpy(dtype=float)

    sp_count = np.isfinite(sp_neighbors).sum(axis=1)
    fl_count = np.isfinite(fl_neighbors).sum(axis=1)

    sp_temporal = np.divide(
        np.nansum(sp_neighbors, axis=1),
        np.maximum(sp_count, 1)
    )
    fl_temporal = np.divide(
        np.nansum(fl_neighbors, axis=1),
        np.maximum(fl_count, 1)
    )

    hybrid_speed = np.where(sp_count > 0, sp_temporal, base_speed)
    hybrid_flow = np.where(fl_count > 0, fl_temporal, base_flow)

    true_speed = actual["speed_kmh"].to_numpy(dtype=float)
    true_flow = actual["flow_vph"].to_numpy(dtype=float)

    lanes = score_task1.lane_vector(
        actual["station_id"], actual["link_id"], lanes_by_station
    )

    for model, pred_speed, pred_flow in [
        ("Historical Mean", base_speed, base_flow),
        ("Temporal Hybrid", hybrid_speed, hybrid_flow)
    ]:
        squared_errors[(model, regime)]["speed"] += float(
            np.sum((pred_speed - true_speed) ** 2)
        )
        squared_errors[(model, regime)]["flow"] += float(
            np.sum(((pred_flow - true_flow) / lanes) ** 2)
        )
        squared_errors[(model, regime)]["n"] += len(actual)

    processed_days += 1

    if day_idx % 25 == 0:
        print(f"Evaluated days: {day_idx}/{len(truth_files)}")

print("\nProcessed days:", processed_days)
print("Skipped days:", len(skipped_days))
if skipped_days:
    print("Skipped examples:", skipped_days[:10])

rows = []

for (model, regime), values in sorted(squared_errors.items()):
    n = values["n"]
    if n == 0:
        continue

    rmse_speed = np.sqrt(values["speed"] / n)
    rmse_flow = np.sqrt(values["flow"] / n)

    s_speed = max(
        0.0, 1.0 - rmse_speed / score_task1.SPEED_NORMALIZER
    )
    s_flow = max(
        0.0, 1.0 - rmse_flow / score_task1.FLOW_NORMALIZER
    )
    s_state = (
        score_task1.SPEED_WEIGHT * s_speed
        + score_task1.FLOW_WEIGHT * s_flow
    )

    rows.append({
        "model": model,
        "regime": regime,
        "n_targets": n,
        "RMSE_speed": rmse_speed,
        "RMSE_flow_per_lane": rmse_flow,
        "S_speed": s_speed,
        "S_flow": s_flow,
        "S_state": s_state
    })

results = pd.DataFrame(rows)
display(results)

summary = (
    results.groupby("model", as_index=False)["S_state"]
    .mean()
    .rename(columns={"S_state": "macro_mean_S_state"})
)
print("\nMacro mean across regimes:")
display(summary)


In [ ]:
baseline_path = Path("/kaggle/working/state_baseline_validation_D12_I5_N.csv")
output_path = Path("/kaggle/working/state_hybrid_validation_D12_I5_N.csv")

submission = pd.read_csv(baseline_path)
submission["timestamp"] = pd.to_datetime(submission["timestamp"], utc=True)
submission["station_id"] = submission["station_id"].astype(str)
submission["link_id"] = submission["link_id"].astype(str)

keys = ["timestamp", "station_id", "link_id"]
masked_root = RELEASE / "corridors" / PANEL / "validation" / "mainline_states_masked"

total_temporal = 0
total_targets = 0
processed_files = 0

for regime in ["R1", "R2", "R3"]:
    regime_dir = masked_root / f"mask_regime={regime}"
    files = sorted(regime_dir.glob("**/*.parquet"))

    print(f"{regime}: {len(files)} masked files")

    for path in files:
        day = path.stem.replace("synthetic_mainline_", "").replace("_", "-")

        targets = submission[
            (submission["mask_regime"] == regime)
            & (submission["timestamp"].dt.strftime("%Y-%m-%d") == day)
        ].copy()

        if targets.empty:
            continue

        targets["_row_index"] = targets.index

        masked = pd.read_parquet(
            path,
            columns=["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph"]
        )

        masked["timestamp"] = pd.to_datetime(masked["timestamp"], utc=True)
        masked["station_id"] = masked["station_id"].astype(str)
        masked["link_id"] = masked["link_id"].astype(str)

        visible = masked[
            masked["speed_kmh"].notna() & masked["flow_vph"].notna()
        ][keys + ["speed_kmh", "flow_vph"]].copy()

        previous = visible.copy()
        previous["timestamp"] += pd.Timedelta(minutes=5)
        previous = previous.rename(columns={
            "speed_kmh": "speed_prev",
            "flow_vph": "flow_prev"
        })

        following = visible.copy()
        following["timestamp"] -= pd.Timedelta(minutes=5)
        following = following.rename(columns={
            "speed_kmh": "speed_next",
            "flow_vph": "flow_next"
        })

        temporal = targets[["_row_index"] + keys].merge(
            previous, on=keys, how="left", validate="one_to_one"
        ).merge(
            following, on=keys, how="left", validate="one_to_one"
        )

        speed_neighbors = temporal[["speed_prev", "speed_next"]].to_numpy(dtype=float)
        flow_neighbors = temporal[["flow_prev", "flow_next"]].to_numpy(dtype=float)

        speed_count = np.isfinite(speed_neighbors).sum(axis=1)
        flow_count = np.isfinite(flow_neighbors).sum(axis=1)

        speed_temporal = np.divide(
            np.nansum(speed_neighbors, axis=1),
            np.maximum(speed_count, 1)
        )
        flow_temporal = np.divide(
            np.nansum(flow_neighbors, axis=1),
            np.maximum(flow_count, 1)
        )

        row_indices = temporal["_row_index"].to_numpy()
        base_speed = submission.loc[row_indices, "speed_kmh"].to_numpy(dtype=float)
        base_flow = submission.loc[row_indices, "flow_vph"].to_numpy(dtype=float)

        use_speed = speed_count > 0
        use_flow = flow_count > 0

        submission.loc[row_indices[use_speed], "speed_kmh"] = speed_temporal[use_speed]
        submission.loc[row_indices[use_flow], "flow_vph"] = flow_temporal[use_flow]

        total_temporal += int((use_speed & use_flow).sum())
        total_targets += len(targets)
        processed_files += 1

print("\nProcessed files:", processed_files)
print("Targets processed:", total_targets)
print("Targets with temporal neighbors:", total_temporal)

missing = submission[["speed_kmh", "flow_vph"]].isna().sum()
print("\nMissing predictions:")
print(missing)

assert len(submission) == len(pd.read_csv(baseline_path))
assert missing.sum() == 0

submission.to_csv(output_path, index=False)
print("\nSaved:", output_path)
print("Submission shape:", submission.shape)


In [ ]:
output_path = Path("/kaggle/working/state_hybrid_validation_D12_I5_N.csv")
submission = pd.read_csv(output_path)
template = pd.read_csv(
    RELEASE / "task1" / PANEL / "validation" / "train" / "sample_submission_state.csv"
) if False else None

expected_columns = [
    "panel", "timestamp", "station_id", "link_id",
    "mask_regime", "speed_kmh", "flow_vph"
]

print("Columns correct:", submission.columns.tolist() == expected_columns)
print("Row count:", len(submission))
print("Duplicate target keys:", int(
    submission.duplicated(
        ["panel", "timestamp", "station_id", "link_id", "mask_regime"]
    ).sum()
))
print("Missing values:")
print(submission[["speed_kmh", "flow_vph"]].isna().sum())

print("\nSpeed summary:")
print(submission["speed_kmh"].describe(percentiles=[0.01, 0.5, 0.99]))

print("\nFlow summary:")
print(submission["flow_vph"].describe(percentiles=[0.01, 0.5, 0.99]))

print("\nNegative predictions:")
print("Speed:", int((submission["speed_kmh"] < 0).sum()))
print("Flow:", int((submission["flow_vph"] < 0).sum()))

print("\nRows by regime:")
print(submission["mask_regime"].value_counts())

In [ ]:
submission_path = Path(
    "/kaggle/working/state_hybrid_validation_D12_I5_N.csv"
)

template_path = (
    RELEASE / "task1" / PANEL / "validation"
    / "sample_submission_state.csv"
)

submission = pd.read_csv(submission_path)
template = pd.read_csv(template_path)

key_cols = [
    "panel",
    "timestamp",
    "station_id",
    "link_id",
    "mask_regime",
]

print("Template exists:", template_path.exists())
print("Template rows:", len(template))
print("Submission rows:", len(submission))

print(
    "Same columns:",
    submission.columns.tolist() == template.columns.tolist()
)

submission_keys = submission[key_cols].copy()
template_keys = template[key_cols].copy()

submission_keys["timestamp"] = pd.to_datetime(
    submission_keys["timestamp"], utc=True
)
template_keys["timestamp"] = pd.to_datetime(
    template_keys["timestamp"], utc=True
)

comparison = template_keys.merge(
    submission_keys,
    on=key_cols,
    how="outer",
    indicator=True,
    validate="one_to_one",
)

print("\nKey comparison:")
print(comparison["_merge"].value_counts())

print(
    "\nPrediction values complete:",
    submission[["speed_kmh", "flow_vph"]].notna().all().all()
)

In [ ]:
task1_root = RELEASE / "task1"

template_paths = sorted(
    task1_root.glob("*/validation/sample_submission_state.csv")
)

total_rows = 0

print("Panel templates found:", len(template_paths))
print("-" * 60)

for path in template_paths:
    template = pd.read_csv(path)
    total_rows += len(template)
    print(f"{path.parent.parent.name}: {len(template):,} rows")

print("-" * 60)
print(f"Total validation rows: {total_rows:,}")
print(f"Competition expected rows: {6_980_503:,}")
print(f"Matches expected: {total_rows == 6_980_503}")

In [ ]:
task1_root = RELEASE / "task1"

for split_dir in sorted(task1_root.iterdir()):
    if not split_dir.is_dir():
        continue

    print(f"\n=== {split_dir.name} ===")

    for path in sorted(split_dir.rglob("sample_submission_state.csv")):
        try:
            rows = sum(1 for _ in open(path, "r", encoding="utf-8")) - 1
            print(f"{path.relative_to(RELEASE)}: {rows:,} rows")
        except Exception as exc:
            print(f"Could not read {path}: {exc}")

print("\n=== Competition release directories ===")
for path in sorted((RELEASE / "corridors").iterdir()):
    if path.is_dir():
        print(path.name)
    

In [ ]:
print("=== Submission templates ===")

for path in sorted(RELEASE.rglob("*sample_submission*")):
    if path.is_file():
        try:
            with open(path, "r", encoding="utf-8") as f:
                rows = sum(1 for _ in f) - 1
            print(f"{path.relative_to(RELEASE)} | {rows:,} rows")
        except Exception as exc:
            print(f"{path.relative_to(RELEASE)} | Error: {exc}")

print("\n=== Instruction files ===")

for root in [RELEASE, REPO]:
    for pattern in ["README*", "*.md", "*.yaml", "*.yml", "*.json"]:
        for path in sorted(root.glob(pattern)):
            if path.is_file():
                print(path)

print("\n=== Competition root ===")
for path in sorted(RELEASE.iterdir()):
    print(f"{'[DIR] ' if path.is_dir() else '[FILE]'} {path.name}")

In [ ]:
print("=== README: Submission instructions ===")
readme_path = RELEASE / "README.md"

with open(readme_path, "r", encoding="utf-8") as f:
    readme_lines = f.readlines()

for i, line in enumerate(readme_lines[:240], start=1):
    print(f"{i}: {line.rstrip()}")

print("\n=== Main submission template ===")
MASTER = RELEASE / "sample_submission.csv"
master = pd.read_csv(MASTER)

print("Shape:", master.shape)
print("Columns:", master.columns.tolist())
print("\nFirst 10 rows:")
display(master.head(10))

print("\nData types:")
print(master.dtypes)

print("\nMissing values:")
print(master.isna().sum())

print("\n=== Submission key ===")
KEY = RELEASE / "submission_key.csv"
key = pd.read_csv(KEY)

print("Shape:", key.shape)
print("Columns:", key.columns.tolist())
display(key.head(10))

print("\n=== Task-specific template columns ===")
for task in ["task1", "task2", "task4"]:
    task_dir = RELEASE / task
    print(f"\n{task.upper()}")

    for path in sorted(task_dir.rglob("sample_submission*"))[:2]:
        sample = pd.read_csv(path, nrows=5)
        print(path.relative_to(RELEASE))
        print("Columns:", sample.columns.tolist())
        display(sample.head(3))

In [ ]:
KEY = RELEASE / "submission_key.csv"
key = pd.read_csv(KEY, low_memory=False)

print("=== Rows by task and split ===")
display(
    key.groupby(["task", "split"])
       .size()
       .unstack(fill_value=0)
)

print("\n=== Rows by task, split, and panel ===")
display(
    key.groupby(["task", "split", "panel"], dropna=False)
       .size()
       .rename("rows")
       .reset_index()
       .sort_values(["task", "split", "panel"])
)

print("\n=== Submission ID integrity ===")
print("Total rows:", len(key))
print("Unique submission IDs:", key["submission_id"].nunique())
print("Missing IDs:", key["submission_id"].isna().sum())
print("Duplicate IDs:", key["submission_id"].duplicated().sum())

print("\n=== Task counts ===")
print(key["task"].value_counts(dropna=False))

print("\n=== Split counts ===")
print(key["split"].value_counts(dropna=False))

In [ ]:
print("=== Task 2: Queue scripts ===")
task2_dir = REPO / "src" / "task2"
if task2_dir.exists():
    for p in sorted(task2_dir.rglob("*.py")):
        print(p)
else:
    print("Task 2 source directory not found")

print("\n=== Task 4: ODME scripts ===")
task4_dir = REPO / "src" / "task4"
if task4_dir.exists():
    for p in sorted(task4_dir.rglob("*.py")):
        print(p)
else:
    print("Task 4 source directory not found")

print("\n=== Task-specific baseline/sample files ===")
for task_dir in ["task1", "task2", "task4"]:
    root = RELEASE / task_dir
    print(f"\n{task_dir}:")
    if root.exists():
        for p in sorted(root.rglob("sample_submission*.csv"))[:12]:
            print(p.relative_to(RELEASE))

print("\n=== Official README files ===")
for p in [REPO / "README.md", RELEASE / "README.md"]:
    if p.exists():
        print(f"\n--- {p} ---")
        print(p.read_text(encoding="utf-8")[:12000])


In [ ]:
scripts = [
    REPO / "src/task1/build_task1_baseline_submission.py",
    REPO / "src/task2/build_task2_persistence_submission.py",
    REPO / "src/task4/build_task4_odme_artifacts.py",
]

for script in scripts:
    print("\n" + "=" * 90)
    print(f"SCRIPT: {script}")
    print("=" * 90)

    if not script.exists():
        print("File not found")
        continue

    lines = script.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines, start=1):
        if (
            "__main__" in line
            or "ArgumentParser" in line
            or "add_argument" in line
            or "def main" in line
            or "def build" in line
            or "output" in line.lower()
            or "argparse" in line
        ):
            start = max(0, i - 3)
            end = min(len(lines), i + 5)
            print(f"\n--- Lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)
REPO = Path("/kaggle/working/trafficflowbench-public")
OUT = Path("/kaggle/working/trafficflow_baselines")

OUT.mkdir(parents=True, exist_ok=True)

def run_script(script, args):
    cmd = [sys.executable, str(script), *map(str, args)]
    print("\nRunning:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# Task 1: Historical-mean baseline for all panels and both scored splits
task1_script = REPO / "src/task1/build_task1_baseline_submission.py"

for split in ["validation", "private"]:
    run_script(
        task1_script,
        [
            "--release-root", RELEASE,
            "--split", split,
            "--output", OUT / f"state_baseline_{split}.csv",
        ],
    )

# Task 2: Official persistence baseline
task2_script = REPO / "src/task2/build_task2_persistence_submission.py"

for split in ["validation", "private"]:
    run_script(
        task2_script,
        [
            "--release-root", RELEASE,
            "--split", split,
            "--output", OUT / f"queue_baseline_{split}.csv",
        ],
    )

# Task 4: Official ODME artifacts
task4_script = REPO / "src/task4/build_task4_odme_artifacts.py"

for split in ["validation", "private"]:
    run_script(
        task4_script,
        [
            "--release-root", RELEASE,
            "--split", split,
            "--output-root", OUT / f"odme_{split}",
        ],
    )

print("\nFinished generating official baseline files.")
print("Output directory:", OUT)


In [ ]:
OUT = Path("/kaggle/working/trafficflow_baselines")

# Inspect Task 1 and Task 2 baseline files
files_to_check = [
    OUT / "state_baseline_validation.csv",
    OUT / "state_baseline_private.csv",
    OUT / "queue_baseline_validation.csv",
    OUT / "queue_baseline_private.csv",
]

for path in files_to_check:
    print("\n" + "=" * 90)
    print("FILE:", path.name)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path, nrows=5)
    print("Columns:", df.columns.tolist())
    print("Sample:")
    display(df)

# List and inspect all ODME CSV files
for split in ["validation", "private"]:
    folder = OUT / f"odme_{split}"

    print("\n" + "=" * 90)
    print(f"ODME FILES — {split.upper()}")

    csv_files = sorted(folder.rglob("*.csv"))

    for path in csv_files:
        print(f"\nFile: {path.relative_to(folder)}")

        sample = pd.read_csv(path, nrows=5)
        print("Columns:", sample.columns.tolist())
        print("Sample:")
        display(sample)

        # Count rows without loading the complete CSV into memory
        with open(path, "rb") as f:
            row_count = sum(1 for _ in f) - 1

        print("Rows:", row_count)
    

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)
OUT = Path("/kaggle/working/trafficflow_baselines")
KEY_PATH = RELEASE / "submission_key.csv"
TEMPLATE_PATH = RELEASE / "sample_submission.csv"
FINAL_PATH = Path("/kaggle/working/submission.csv")

# Load only the key columns required for joining predictions
key_columns = [
    "submission_id", "task", "split", "panel", "timestamp",
    "station_id", "link_id", "mask_regime", "window_id",
    "departure_time", "path_id", "origin_zone", "destination_zone"
]

key = pd.read_csv(
    KEY_PATH,
    usecols=key_columns,
    dtype={c: "string" for c in key_columns}
)

submission = pd.read_csv(
    TEMPLATE_PATH,
    dtype={"submission_id": "string"}
)

assert len(key) == 6_980_503, f"Unexpected key rows: {len(key):,}"
assert key["submission_id"].is_unique, "Duplicate IDs in submission key"
assert submission["submission_id"].is_unique, "Duplicate IDs in template"
assert set(key["submission_id"]) == set(submission["submission_id"]), (
    "Template IDs and key IDs do not match"
)

# Keep the original submission template and its row order
submission = submission.set_index("submission_id")
key = key.set_index("submission_id", drop=False)

def read_split_files(prefix, split, folder=None):
    if folder is None:
        path = OUT / f"{prefix}_{split}.csv"
    else:
        path = OUT / folder / "baseline_submission.csv"

    if not path.exists():
        raise FileNotFoundError(f"Missing expected file: {path}")

    df = pd.read_csv(path)
    df["split"] = split
    return df

state_parts = [
    read_split_files("state_baseline", split)
    for split in ["validation", "private"]
]
queue_parts = [
    read_split_files("queue_baseline", split)
    for split in ["validation", "private"]
]
odme_parts = [
    read_split_files("", split, folder=f"odme_{split}")
    for split in ["validation", "private"]
]

state_pred = pd.concat(state_parts, ignore_index=True)
queue_pred = pd.concat(queue_parts, ignore_index=True)
odme_pred = pd.concat(odme_parts, ignore_index=True)

# Normalize timestamps to UTC before joining
key["timestamp"] = pd.to_datetime(key["timestamp"], utc=True, errors="coerce")
state_pred["timestamp"] = pd.to_datetime(
    state_pred["timestamp"], utc=True, errors="coerce"
)
queue_pred["timestamp"] = pd.to_datetime(
    queue_pred["timestamp"], utc=True, errors="coerce"
)

def attach_predictions(task_name, predictions, join_columns, prediction_columns):
    task_key = key.loc[key["task"] == task_name].copy()

    if task_key.empty:
        raise ValueError(f"No rows found for task={task_name!r}")

    if predictions.duplicated(join_columns).any():
        raise ValueError(
            f"Duplicate join keys found in predictions for {task_name}"
        )

    merged = task_key.merge(
        predictions[join_columns + prediction_columns],
        on=join_columns,
        how="left",
        validate="one_to_one",
        indicator=True
    )

    unmatched = int((merged["_merge"] != "both").sum())
    if unmatched:
        raise ValueError(
            f"{task_name}: {unmatched:,} key rows did not match predictions"
        )

    if len(merged) != len(task_key):
        raise ValueError(f"{task_name}: row count changed during merge")

    if merged[prediction_columns].isna().any().any():
        raise ValueError(f"{task_name}: missing prediction values")

    # Assign by submission_id, preserving the original template order
    for column in prediction_columns:
        values = merged.set_index("submission_id")[column]
        submission.loc[values.index, column] = values

    print(
        f"{task_name}: matched {len(merged):,} / "
        f"{len(task_key):,} rows"
    )

# Task 1: speed and flow
attach_predictions(
    "state",
    state_pred,
    [
        "split", "panel", "timestamp", "station_id",
        "link_id", "mask_regime"
    ],
    ["speed_kmh", "flow_vph"]
)

# Task 2: queue classification
attach_predictions(
    "queue",
    queue_pred,
    ["split", "window_id", "timestamp", "link_id"],
    ["queue_pred"]
)

# Task 4: ODME path-flow estimation
attach_predictions(
    "odme",
    odme_pred,
    [
        "split", "panel", "departure_time", "path_id",
        "origin_zone", "destination_zone"
    ],
    ["path_flow"]
)

# Restore the submission_id column
submission = submission.reset_index()

# Final integrity checks
expected_rows = 6_980_503
assert len(submission) == expected_rows
assert submission["submission_id"].is_unique
assert submission["submission_id"].notna().all()

prediction_columns = [
    "speed_kmh", "flow_vph", "queue_pred", "path_flow"
]

for column in prediction_columns:
    submission[column] = pd.to_numeric(
        submission[column], errors="raise"
    )
    assert np.isfinite(submission[column]).all(), (
        f"Non-finite values found in {column}"
    )

assert submission["queue_pred"].isin([0, 1]).all(), (
    "Queue predictions must be binary"
)
assert (submission["speed_kmh"] >= 0).all()
assert (submission["flow_vph"] >= 0).all()
assert (submission["path_flow"] >= 0).all()

# Ensure unrelated task columns remain zero
task_masks = {
    "state": submission["task"].eq("state"),
    "queue": submission["task"].eq("queue"),
    "odme": submission["task"].eq("odme")
}

for task_name, mask in task_masks.items():
    assert mask.any(), f"No submission rows found for {task_name}"

    other_columns = {
        "state": ["queue_pred", "path_flow"],
        "queue": ["speed_kmh", "flow_vph", "path_flow"],
        "odme": ["speed_kmh", "flow_vph", "queue_pred"]
    }[task_name]

    for column in other_columns:
        assert (submission.loc[mask, column] == 0).all(), (
            f"Unexpected nonzero values in {column} for {task_name}"
        )

# Save in the exact column order of the official template
template_columns = pd.read_csv(TEMPLATE_PATH, nrows=0).columns.tolist()
submission = submission[template_columns]
submission.to_csv(FINAL_PATH, index=False)

print("\n" + "=" * 60)
print("FINAL SUBMISSION VALIDATION PASSED")
print("Rows:", f"{len(submission):,}")
print("Unique submission IDs:", f"{submission['submission_id'].nunique():,}")
print("Task counts:")
print(submission["task"].value_counts().to_string())
print("File:", FINAL_PATH)
print("Size (MB):", round(FINAL_PATH.stat().st_size / (1024**2), 2))

In [ ]:
OUT = Path("/kaggle/working/trafficflow_baselines")

for split in ["validation", "private"]:
    path = OUT / f"odme_{split}" / "baseline_submission.csv"
    print(f"{split}: {path.exists()} | {path}")

In [ ]:
path = Path("/kaggle/working/submission.csv")

print("File exists:", path.exists())

if path.exists():
    print("File size (MB):", round(path.stat().st_size / (1024**2), 2))
    df = pd.read_csv(path, nrows=5)
    print("Columns:", df.columns.tolist())
    print(df.to_string(index=False))

In [ ]:
path = Path("/kaggle/working/submission.csv")

df = pd.read_csv(path)

print("=" * 50)
print("FINAL SUBMISSION AUDIT")
print("=" * 50)

print("\n1. File")
print("Exists:", path.exists())
print("Size (MB):", round(path.stat().st_size / (1024**2), 2))
print("Rows:", f"{len(df):,}")
print("Columns:", df.columns.tolist())

print("\n2. Task counts")
print(df["task"].value_counts().to_string())

print("\n3. Missing values")
print(df.isna().sum().to_string())

print("\n4. Duplicate submission IDs")
print(df["submission_id"].duplicated().sum())

print("\n5. Prediction ranges")
for col in ["speed_kmh", "flow_vph", "queue_pred", "path_flow"]:
    print(
        f"{col}: min={df[col].min()}, "
        f"max={df[col].max()}, "
        f"unique={df[col].nunique()}"
    )

print("\n6. Non-finite values")
for col in ["speed_kmh", "flow_vph", "queue_pred", "path_flow"]:
    values = pd.to_numeric(df[col], errors="coerce")
    print(f"{col}: {(~np.isfinite(values)).sum():,}")

print("\n7. Nonzero predictions by task")
for task in ["state", "queue", "odme"]:
    subset = df[df["task"] == task]
    print(f"\n{task}:")
    for col in ["speed_kmh", "flow_vph", "queue_pred", "path_flow"]:
        print(f"  {col}: {(subset[col] != 0).sum():,}")

print("\nAUDIT COMPLETE")

In [ ]:
path = "/kaggle/working/submission.csv"
df = pd.read_csv(path)

queue = df.loc[df["task"] == "queue", "queue_pred"]
odme = df.loc[df["task"] == "odme", "path_flow"]

print("QUEUE PREDICTION DISTRIBUTION")
print(queue.value_counts(dropna=False).sort_index())
print("Queue positive rate:", round(queue.mean(), 5))

print("\nODME PATH FLOW SUMMARY")
print(odme.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]))
print("Zero path flows:", int((odme == 0).sum()))
print("Positive path flows:", int((odme > 0).sum()))

print("\nSTATE SPEED SUMMARY")
print(
    df.loc[df["task"] == "state", "speed_kmh"]
    .describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

print("Queue-related training files:\n")

for p in RELEASE.rglob("*"):
    if p.is_file() and any(
        word in p.name.lower()
        for word in ["queue", "window_history", "window_index"]
    ):
        print(p)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panel = "D7_I405_S"
train_dir = RELEASE / "task2" / panel / "train"

for filename in [
    "window_index.csv",
    "window_history.parquet",
    "sample_submission_queue.csv",
]:
    path = train_dir / filename
    print("\n" + "=" * 70)
    print("FILE:", filename)

    if filename.endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))
    print("\nData types:")
    print(df.dtypes.to_string())

In [ ]:
REPO = Path("/kaggle/working/trafficflowbench-public")

print("Task 2 files in repository:\n")

for p in sorted((REPO / "src" / "task2").rglob("*")):
    if p.is_file():
        print(p)

print("\nQueue-related files in competition data:\n")

RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

for p in sorted((RELEASE / "task2" / "D7_I405_S" / "train").iterdir()):
    if p.is_file():
        print(p.name)

In [ ]:
REPO = Path("/kaggle/working/trafficflowbench-public")
files = [
    REPO / "src/task2/queue_utils.py",
    REPO / "src/task2/score_task2.py",
    REPO / "src/task2/build_task2_persistence_submission.py",
]

for path in files:
    print("\n" + "=" * 90)
    print(f"FILE: {path.name}")
    print("=" * 90)

    content = path.read_text(encoding="utf-8")
    print(content)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panel = "D7_I405_S"
train_dir = RELEASE / "task2" / panel / "train"

windows = pd.read_csv(train_dir / "window_index.csv")
history = pd.read_parquet(train_dir / "window_history.parquet")

history["timestamp"] = pd.to_datetime(history["timestamp"], utc=True)
history = history.merge(
    windows[["window_id", "condition", "forecast_origin"]],
    on="window_id",
    how="left",
    validate="many_to_one"
)

history["forecast_origin"] = pd.to_datetime(
    history["forecast_origin"], utc=True
)

last_time = history.groupby("window_id")["timestamp"].transform("max")
last = history.loc[history["timestamp"].eq(last_time)].copy()

print("Windows:", windows.shape)
print("Last-history rows:", last.shape)
print("\nCondition distribution:")
print(windows["condition"].value_counts(dropna=False))

print("\nLast observed data summary by condition:")
print(
    last.groupby("condition")[
        ["speed_kmh", "flow_vph", "occupancy", "pct_observed"]
    ].agg(["count", "mean", "median"]).round(3).to_string()
)

print("\nEligibility distribution:")
print(last["is_score_eligible"].value_counts(dropna=False))

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panel = "D7_I405_S"
train_dir = RELEASE / "task2" / panel / "train"

windows = pd.read_csv(train_dir / "window_index.csv")
history = pd.read_parquet(train_dir / "window_history.parquet")

history["timestamp"] = pd.to_datetime(history["timestamp"], utc=True)

# Keep the original eligibility information.
history = history.merge(
    windows[["window_id", "condition"]],
    on="window_id",
    how="left",
    validate="many_to_one"
)

# Inspect the released network files and threshold parameters.
network_dir = RELEASE / "corridors" / panel / "network"

print("Network directory exists:", network_dir.exists())
if network_dir.exists():
    print("Network files:")
    for path in sorted(network_dir.glob("*")):
        print(" -", path.name)

fd_path = network_dir / "fd_parameters.csv"
if fd_path.exists():
    fd = pd.read_csv(fd_path)
    print("\nFD parameter columns:", fd.columns.tolist())
    print(fd.head().to_string(index=False))

# Use only rows with an observed speed for trend analysis.
observed = history.dropna(subset=["speed_kmh"]).copy()
observed = observed.sort_values(["window_id", "link_id", "timestamp"])

# Last three observed readings for each window-link pair.
recent = (
    observed.groupby(["window_id", "link_id"], group_keys=False)
    .tail(3)
    .copy()
)

recent["reading_number"] = (
    recent.groupby(["window_id", "link_id"]).cumcount() + 1
)

wide = recent.pivot_table(
    index=["window_id", "link_id", "condition"],
    columns="reading_number",
    values=["speed_kmh", "occupancy"],
    aggfunc="last"
)

wide.columns = [f"{feature}_t{step}" for feature, step in wide.columns]
wide = wide.reset_index()

# Calculate changes only when both endpoints are available.
if {"speed_kmh_t1", "speed_kmh_t3"}.issubset(wide.columns):
    wide["speed_change"] = wide["speed_kmh_t3"] - wide["speed_kmh_t1"]

if {"occupancy_t1", "occupancy_t3"}.issubset(wide.columns):
    wide["occupancy_change"] = (
        wide["occupancy_t3"] - wide["occupancy_t1"]
    )

print("\nRecent-reading coverage:")
print("Window-link pairs:", len(wide))
print(
    "Pairs with 3 readings:",
    int(wide.get("speed_kmh_t3", pd.Series(dtype=float)).notna().sum())
)

metrics = [
    c for c in [
        "speed_kmh_t3",
        "speed_change",
        "occupancy_t3",
        "occupancy_change"
    ]
    if c in wide.columns
]

print("\nRecent features by window condition:")
print(
    wide.groupby("condition")[metrics]
    .agg(["count", "mean", "median"])
    .round(3)
    .to_string()
)

print("\nSpeed trend counts:")
if "speed_change" in wide.columns:
    print(
        wide.assign(
            trend=np.select(
                [
                    wide["speed_change"] < -5,
                    wide["speed_change"] > 5
                ],
                ["falling", "rising"],
                default="stable"
            )
        )
        .groupby(["condition", "trend"])
        .size()
        .to_string()
    )

In [ ]:
REPO = Path("/kaggle/working/trafficflowbench-public")
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

sys.path.insert(0, str(REPO / "src"))

from task2.queue_utils import thresholds

panel = "D7_I405_S"
panel_dir = RELEASE / "corridors" / panel
links_path = panel_dir / "network" / "links.csv"
fd_path = panel_dir / "network" / "fd_parameters.csv"

links = pd.read_csv(links_path)
fd = pd.read_csv(fd_path)

print("Links shape:", links.shape)
print("Links columns:", links.columns.tolist())
print("\nFD columns:", fd.columns.tolist())

threshold_map, threshold_source = thresholds(panel_dir, links)

print("\nThreshold source:", threshold_source)
print("Number of thresholds:", len(threshold_map))

thresholds_df = pd.DataFrame(
    threshold_map.items(),
    columns=["link_id", "queue_speed_threshold"]
)

thresholds_df["link_id"] = thresholds_df["link_id"].astype(str)

comparison = (
    links.assign(link_id=links["link_id"].astype(str))
    .merge(thresholds_df, on="link_id", how="left")
)

print("\nThreshold summary:")
print(
    comparison["queue_speed_threshold"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .round(3)
)

print("\nThreshold examples:")
print(
    comparison[
        ["link_id", "free_speed_kmh", "queue_speed_threshold"]
    ].head(15).to_string(index=False)
)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panel = "D7_I405_S"
train_dir = RELEASE / "task2" / panel / "train"
network_dir = RELEASE / "corridors" / panel / "network"

windows = pd.read_csv(train_dir / "window_index.csv")
history = pd.read_parquet(train_dir / "window_history.parquet")
links = pd.read_csv(network_dir / "links.csv")

history["timestamp"] = pd.to_datetime(history["timestamp"], utc=True)

# Keep only the last historical timestamp for each window.
last_timestamp = history.groupby("window_id")["timestamp"].transform("max")
last = history.loc[history["timestamp"].eq(last_timestamp)].copy()

last["link_id"] = last["link_id"].astype(str)
links["link_id"] = links["link_id"].astype(str)

last = last.merge(
    links[["link_id", "free_speed_kmh", "capacity_vph"]],
    on="link_id",
    how="left",
    validate="many_to_one"
)

last = last.merge(
    windows[["window_id", "condition"]],
    on="window_id",
    how="left",
    validate="many_to_one"
)

last["speed_threshold"] = last["free_speed_kmh"] * 0.6
last["queue_by_speed"] = (
    last["speed_kmh"].notna()
    & (last["speed_kmh"] <= last["speed_threshold"])
).astype(int)

eligible = last.loc[last["is_score_eligible"].astype(bool)].copy()

print("Eligible last-history rows:", len(eligible))
print("Rows with observed speed:", int(eligible["speed_kmh"].notna().sum()))
print("Rows with observed occupancy:", int(eligible["occupancy"].notna().sum()))

print("\nSpeed-based queue rate by window condition:")
print(
    eligible.groupby("condition")["queue_by_speed"]
    .agg(["count", "mean", "sum"])
    .round(4)
)

print("\nSpeed and occupancy relationship:")
print(
    eligible[["speed_kmh", "occupancy", "pct_observed"]]
    .corr()
    .round(3)
)

print("\nSpeed-based queue rate by occupancy band:")
eligible["occupancy_band"] = pd.cut(
    eligible["occupancy"],
    bins=[-np.inf, 0.10, 0.20, 0.30, 0.50, np.inf],
    labels=["<=0.10", "0.10-0.20", "0.20-0.30", "0.30-0.50", ">0.50"]
)

print(
    eligible.groupby("occupancy_band", observed=False)
    .agg(
        rows=("link_id", "size"),
        mean_speed=("speed_kmh", "mean"),
        speed_queue_rate=("queue_by_speed", "mean"),
        mean_occupancy=("occupancy", "mean")
    )
    .round(3)
    .to_string()
)

In [ ]:
# Reuse the eligible dataframe from the previous cell.
eligible = eligible.copy()

free_speed = pd.to_numeric(
    eligible["free_speed_kmh"], errors="coerce"
)
speed = pd.to_numeric(
    eligible["speed_kmh"], errors="coerce"
)

for ratio in [0.60, 0.70, 0.80]:
    col = f"queue_rate_at_{int(ratio * 100)}pct"
    eligible[col] = (
        speed.notna()
        & free_speed.notna()
        & (speed <= ratio * free_speed)
    ).astype(int)

rate_cols = [
    "queue_rate_at_60pct",
    "queue_rate_at_70pct",
    "queue_rate_at_80pct",
]

print("Overall diagnostic rates:")
print(eligible[rate_cols].mean().round(4).to_string())

print("\nRates by window condition:")
print(
    eligible.groupby("condition")[rate_cols]
    .mean()
    .round(4)
    .to_string()
)

print("\nPositive counts by condition:")
print(
    eligible.groupby("condition")[rate_cols]
    .sum()
    .astype(int)
    .to_string()
)

In [ ]:
eligible = eligible.copy()

eligible["speed_threshold"] = (
    pd.to_numeric(eligible["free_speed_kmh"], errors="coerce") * 0.60
)

eligible["speed_margin"] = (
    pd.to_numeric(eligible["speed_kmh"], errors="coerce")
    - eligible["speed_threshold"]
)

eligible["threshold_band"] = pd.cut(
    eligible["speed_margin"],
    bins=[-np.inf, -20, -10, -5, 0, 5, 10, 20, np.inf],
    labels=[
        "< -20",
        "-20 to -10",
        "-10 to -5",
        "-5 to 0",
        "0 to 5",
        "5 to 10",
        "10 to 20",
        "> 20"
    ],
    right=False
)

print("Speed margin = observed speed - official threshold")
print("\nMargin summary:")
print(
    eligible["speed_margin"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
)

print("\nCounts by distance from threshold:")
print(
    eligible["threshold_band"]
    .value_counts(sort=False, dropna=False)
    .to_string()
)

print("\nMargin summary by condition:")
print(
    eligible.groupby("condition", observed=False)["speed_margin"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
    .to_string()
)

In [ ]:
margin = pd.to_numeric(
    eligible["speed_margin"], errors="coerce"
).dropna()

print("Number of valid margins:", len(margin))

print("\nQuantiles calculated directly:")
print(
    margin.quantile(
        [0.00, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    ).to_string()
)

print("\nThreshold classification:")
print("Below threshold:", int((margin <= 0).sum()))
print("Above threshold:", int((margin > 0).sum()))
print("Within 5 km/h of threshold:", int(margin.abs().le(5).sum()))
print("Within 10 km/h of threshold:", int(margin.abs().le(10).sum()))

print("\nClassification by condition:")
print(
    eligible.assign(
        below_threshold=eligible["speed_margin"].le(0)
    )
    .groupby("condition")["below_threshold"]
    .agg(["count", "sum", "mean"])
    .to_string()
)

In [ ]:
# Reuse the eligible last-history data and the training window index.
queue_by_window = (
    eligible.groupby(["window_id", "condition"], as_index=False)
    .agg(
        eligible_links=("link_id", "size"),
        predicted_queue_links=("queue_by_speed", "sum"),
        queue_rate=("queue_by_speed", "mean"),
        mean_speed=("speed_kmh", "mean"),
        mean_occupancy=("occupancy", "mean"),
    )
)

print("Window-level diagnostic summary:")
print(
    queue_by_window[
        ["predicted_queue_links", "queue_rate", "mean_speed", "mean_occupancy"]
    ]
    .describe()
    .round(3)
    .to_string()
)

print("\nQueue-positive links per window:")
print(
    queue_by_window[
        ["window_id", "condition", "eligible_links", "predicted_queue_links", "queue_rate"]
    ]
    .sort_values(["condition", "queue_rate"])
    .to_string(index=False)
)

print("\nWindows with zero predicted queue links:")
print(
    int((queue_by_window["predicted_queue_links"] == 0).sum()),
    "of",
    len(queue_by_window)
)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panel = "D7_I405_S"
train_dir = RELEASE / "task2" / panel / "train"

template = pd.read_csv(train_dir / "sample_submission_queue.csv")
windows = pd.read_csv(train_dir / "window_index.csv")

template["timestamp"] = pd.to_datetime(
    template["timestamp"], utc=True
)

print("Template shape:", template.shape)
print("Columns:", template.columns.tolist())

print("\nRows per window:")
print(
    template.groupby("window_id")
    .agg(
        rows=("link_id", "size"),
        unique_links=("link_id", "nunique"),
        unique_timestamps=("timestamp", "nunique"),
        first_time=("timestamp", "min"),
        last_time=("timestamp", "max"),
    )
    .to_string()
)

print("\nForecast metadata:")
print(
    windows[
        [
            "window_id",
            "forecast_origin",
            "forecast_start",
            "forecast_end",
            "history_coverage",
            "future_coverage",
        ]
    ].to_string(index=False)
)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panel = "D7_I405_S"
train_dir = RELEASE / "task2" / panel / "train"
network_dir = RELEASE / "corridors" / panel / "network"

windows = pd.read_csv(train_dir / "window_index.csv")
history = pd.read_parquet(train_dir / "window_history.parquet")
links = pd.read_csv(network_dir / "links.csv")

history["timestamp"] = pd.to_datetime(history["timestamp"], utc=True)
history["link_id"] = history["link_id"].astype(str)
links["link_id"] = links["link_id"].astype(str)

history = history.merge(
    links[["link_id", "free_speed_kmh"]],
    on="link_id",
    how="left",
    validate="many_to_one"
)

history["speed_threshold"] = history["free_speed_kmh"] * 0.60

history["queue_by_speed"] = (
    history["speed_kmh"].notna()
    & (history["speed_kmh"] <= history["speed_threshold"])
).astype(int)

history = history.sort_values(
    ["window_id", "link_id", "timestamp"]
).copy()

# Calculate historical queue persistence for each window-link pair.
history["observed_speed"] = history["speed_kmh"].notna().astype(int)

history["queue_observed"] = history["queue_by_speed"].where(
    history["observed_speed"].eq(1)
)

group_cols = ["window_id", "link_id"]

history["recent_3_count"] = (
    history.groupby(group_cols)["observed_speed"]
    .transform(lambda s: s.rolling(3, min_periods=1).sum())
)

history["recent_3_queue_count"] = (
    history.groupby(group_cols)["queue_observed"]
    .transform(lambda s: s.rolling(3, min_periods=1).sum())
)

history["recent_6_count"] = (
    history.groupby(group_cols)["observed_speed"]
    .transform(lambda s: s.rolling(6, min_periods=1).sum())
)

history["recent_6_queue_count"] = (
    history.groupby(group_cols)["queue_observed"]
    .transform(lambda s: s.rolling(6, min_periods=1).sum())
)

last_ts = history.groupby("window_id")["timestamp"].transform("max")
last = history.loc[history["timestamp"].eq(last_ts)].copy()

last["queue_share_3"] = (
    last["recent_3_queue_count"] / last["recent_3_count"].replace(0, np.nan)
)

last["queue_share_6"] = (
    last["recent_6_queue_count"] / last["recent_6_count"].replace(0, np.nan)
)

last = last.merge(
    windows[["window_id", "condition"]],
    on="window_id",
    how="left",
    validate="many_to_one"
)

eligible = last.loc[last["is_score_eligible"].astype(bool)].copy()

print("Eligible rows:", len(eligible))

print("\nHistorical queue persistence by condition:")
print(
    eligible.groupby("condition")[
        ["queue_by_speed", "queue_share_3", "queue_share_6"]
    ]
    .agg(["count", "mean", "median"])
    .round(3)
    .to_string()
)

print("\nLast reading vs recent history:")
print(
    eligible.groupby(
        ["condition", "queue_by_speed"]
    ).agg(
        links=("link_id", "size"),
        mean_recent_3=("queue_share_3", "mean"),
        mean_recent_6=("queue_share_6", "mean"),
    ).round(3).to_string()
)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panels = [
    "D7_I405_S", "D7_I405_N",
    "D7_I10_E", "D7_I10_W",
    "D7_I210_W", "D7_I210_E",
    "D12_I5_N", "D12_I5_S"
]

results = []

for panel in panels:
    train_dir = RELEASE / "task2" / panel / "train"
    network_dir = RELEASE / "corridors" / panel / "network"

    if not train_dir.exists():
        continue

    history_path = train_dir / "window_history.parquet"
    windows_path = train_dir / "window_index.csv"
    links_path = network_dir / "links.csv"

    if not all(p.exists() for p in [history_path, windows_path, links_path]):
        print("Skipping incomplete panel:", panel)
        continue

    history = pd.read_parquet(history_path)
    windows = pd.read_csv(windows_path)
    links = pd.read_csv(links_path)

    history["timestamp"] = pd.to_datetime(history["timestamp"], utc=True)
    history["link_id"] = history["link_id"].astype(str)
    links["link_id"] = links["link_id"].astype(str)

    history = history.merge(
        links[["link_id", "free_speed_kmh"]],
        on="link_id",
        how="left",
        validate="many_to_one"
    )

    history["threshold"] = history["free_speed_kmh"] * 0.60
    history["queue_by_speed"] = (
        history["speed_kmh"].notna()
        & (history["speed_kmh"] <= history["threshold"])
    ).astype(int)

    last_ts = history.groupby("window_id")["timestamp"].transform("max")
    last = history.loc[history["timestamp"].eq(last_ts)].copy()
    eligible = last.loc[last["is_score_eligible"].astype(bool)]

    results.append({
        "panel": panel,
        "windows": windows["window_id"].nunique(),
        "last_history_rows": len(last),
        "eligible_rows": len(eligible),
        "observed_speed_rows": int(eligible["speed_kmh"].notna().sum()),
        "predicted_queue_links": int(eligible["queue_by_speed"].sum()),
        "queue_rate": float(eligible["queue_by_speed"].mean())
            if len(eligible) else np.nan,
        "history_coverage_mean": float(windows["history_coverage"].mean()),
    })

summary = pd.DataFrame(results)

print(summary.round(4).to_string(index=False))
print("\nTotal panels:", len(summary))
print("Total windows:", summary["windows"].sum())
print("Total eligible last-history rows:", summary["eligible_rows"].sum())

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panels = [
    "D7_I405_S", "D7_I405_N",
    "D7_I10_E", "D7_I10_W",
    "D7_I210_W", "D7_I210_E",
    "D12_I5_N", "D12_I5_S"
]

results = []

for panel in panels:
    train_dir = RELEASE / "task2" / panel / "train"
    history = pd.read_parquet(train_dir / "window_history.parquet")
    windows = pd.read_csv(train_dir / "window_index.csv")
    links = pd.read_csv(
        RELEASE / "corridors" / panel / "network" / "links.csv"
    )

    history["timestamp"] = pd.to_datetime(history["timestamp"], utc=True)
    history["link_id"] = history["link_id"].astype(str)
    links["link_id"] = links["link_id"].astype(str)

    history = history.merge(
        links[["link_id", "free_speed_kmh"]],
        on="link_id",
        how="left",
        validate="many_to_one"
    )

    history["queue_by_speed"] = (
        history["speed_kmh"].notna()
        & (
            history["speed_kmh"]
            <= 0.60 * history["free_speed_kmh"]
        )
    ).astype(int)

    last_ts = history.groupby("window_id")["timestamp"].transform("max")
    last = history.loc[history["timestamp"].eq(last_ts)].copy()

    last = last.loc[last["is_score_eligible"].astype(bool)]

    last = last.merge(
        windows[["window_id", "condition"]],
        on="window_id",
        how="left",
        validate="many_to_one"
    )

    summary = (
        last.groupby("condition")
        .agg(
            eligible_links=("link_id", "size"),
            predicted_queue_links=("queue_by_speed", "sum"),
            queue_rate=("queue_by_speed", "mean"),
            mean_speed=("speed_kmh", "mean"),
        )
        .reset_index()
    )

    summary.insert(0, "panel", panel)
    results.append(summary)

condition_summary = pd.concat(results, ignore_index=True)

print("Queue prediction behavior by panel and window condition:")
print(condition_summary.round(4).to_string(index=False))

print("\nOverall summary by condition:")
overall = (
    condition_summary.groupby("condition")
    .agg(
        eligible_links=("eligible_links", "sum"),
        predicted_queue_links=("predicted_queue_links", "sum"),
        mean_panel_queue_rate=("queue_rate", "mean"),
    )
    .reset_index()
)
print(overall.round(4).to_string(index=False))

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panels = [
    "D7_I405_S", "D7_I405_N",
    "D7_I10_E", "D7_I10_W",
    "D7_I210_W", "D7_I210_E",
    "D12_I5_N", "D12_I5_S"
]

results = []

for panel in panels:
    train_dir = RELEASE / "task2" / panel / "train"

    history = pd.read_parquet(train_dir / "window_history.parquet")
    links = pd.read_csv(
        RELEASE / "corridors" / panel / "network" / "links.csv"
    )

    history["timestamp"] = pd.to_datetime(
        history["timestamp"], utc=True
    )
    history["link_id"] = history["link_id"].astype(str)
    links["link_id"] = links["link_id"].astype(str)

    history = history.merge(
        links[["link_id", "free_speed_kmh"]],
        on="link_id",
        how="left",
        validate="many_to_one"
    )

    history = history.sort_values(
        ["window_id", "link_id", "timestamp"]
    ).copy()

    history["below_threshold"] = (
        history["speed_kmh"].notna()
        & (
            history["speed_kmh"]
            <= 0.60 * history["free_speed_kmh"]
        )
    ).astype(float)

    # Missing speeds must not count as non-queue readings.
    history.loc[
        history["speed_kmh"].isna(),
        "below_threshold"
    ] = np.nan

    grouped = history.groupby(
        ["window_id", "link_id"], sort=False
    )["below_threshold"]

    history["recent3_share"] = grouped.transform(
        lambda s: s.rolling(3, min_periods=1).mean()
    )
    history["recent3_count"] = grouped.transform(
        lambda s: s.rolling(3, min_periods=1).count()
    )

    last_ts = history.groupby("window_id")["timestamp"].transform("max")
    last = history.loc[history["timestamp"].eq(last_ts)].copy()

    last = last.loc[
        last["is_score_eligible"].astype(bool)
        & last["speed_kmh"].notna()
    ].copy()

    # Rule 1: current reading
    last["last_reading"] = last["below_threshold"].astype(int)

    # Rule 2: majority of available readings among the last three
    last["recent_majority"] = (
        last["recent3_share"] > 0.5
    ).astype(int)

    # Rule 3: at least one queue reading among the last three
    last["recent_any"] = (
        last["recent3_share"] > 0
    ).astype(int)

    for rule in ["last_reading", "recent_majority", "recent_any"]:
        results.append({
            "panel": panel,
            "rule": rule,
            "eligible_links": len(last),
            "predicted_queue_links": int(last[rule].sum()),
            "queue_rate_pct": 100 * last[rule].mean(),
        })

comparison = pd.DataFrame(results)

print(comparison.round(3).to_string(index=False))

print("\nPivot: predicted queue rate (%)")
print(
    comparison.pivot(
        index="panel",
        columns="rule",
        values="queue_rate_pct"
    ).round(2).to_string()
)

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panels = [
    "D7_I405_S", "D7_I405_N",
    "D7_I10_E", "D7_I10_W",
    "D7_I210_W", "D7_I210_E",
    "D12_I5_N", "D12_I5_S"
]

results = []

for panel in panels:
    path = (
        RELEASE / "task2" / panel / "train"
        / "window_history.parquet"
    )

    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    last_ts = df.groupby("window_id")["timestamp"].transform("max")
    last = df.loc[
        df["timestamp"].eq(last_ts)
        & df["is_score_eligible"].astype(bool)
    ].copy()

    row = {"panel": panel, "eligible_rows": len(last)}

    for col in ["speed_kmh", "flow_vph", "occupancy", "pct_observed"]:
        if col in last.columns:
            row[f"{col}_available_pct"] = (
                100 * last[col].notna().mean()
            )
            row[f"{col}_unique"] = last[col].nunique(dropna=True)

    results.append(row)

availability = pd.DataFrame(results)

print(availability.round(2).to_string(index=False))

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panels = [
    "D7_I405_S", "D7_I405_N",
    "D7_I10_E", "D7_I10_W",
    "D7_I210_W", "D7_I210_E",
    "D12_I5_N", "D12_I5_S"
]

results = []

for panel in panels:
    train_dir = RELEASE / "task2" / panel / "train"

    df = pd.read_parquet(train_dir / "window_history.parquet")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

    last_ts = df.groupby("window_id")["timestamp"].transform("max")

    last = df.loc[
        df["timestamp"].eq(last_ts)
        & df["is_score_eligible"].astype(bool)
    ].copy()

    for col in ["speed_kmh", "flow_vph", "occupancy"]:
        last[col] = pd.to_numeric(last[col], errors="coerce")

    results.append({
        "panel": panel,
        "rows": len(last),
        "corr_speed_occupancy": last["speed_kmh"].corr(
            last["occupancy"]
        ),
        "corr_speed_flow": last["speed_kmh"].corr(
            last["flow_vph"]
        ),
        "corr_occupancy_flow": last["occupancy"].corr(
            last["flow_vph"]
        ),
        "occupancy_p50": last["occupancy"].median(),
        "occupancy_p90": last["occupancy"].quantile(0.90),
        "speed_p10": last["speed_kmh"].quantile(0.10),
        "speed_p50": last["speed_kmh"].median(),
    })

corr_summary = pd.DataFrame(results)

print(corr_summary.round(4).to_string(index=False))

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panels = [
    "D7_I405_S", "D7_I405_N",
    "D7_I10_E", "D7_I10_W",
    "D7_I210_W", "D7_I210_E",
    "D12_I5_N", "D12_I5_S"
]

results = []

for panel in panels:
    train_dir = RELEASE / "task2" / panel / "train"
    history = pd.read_parquet(train_dir / "window_history.parquet")
    links = pd.read_csv(
        RELEASE / "corridors" / panel / "network" / "links.csv"
    )

    history["timestamp"] = pd.to_datetime(
        history["timestamp"], utc=True
    )
    history["link_id"] = history["link_id"].astype(str)
    links["link_id"] = links["link_id"].astype(str)

    history = history.merge(
        links[["link_id", "free_speed_kmh"]],
        on="link_id",
        how="left",
        validate="many_to_one"
    )

    last_ts = history.groupby("window_id")["timestamp"].transform("max")
    last = history.loc[
        history["timestamp"].eq(last_ts)
        & history["is_score_eligible"].astype(bool)
    ].copy()

    last = last.dropna(
        subset=["speed_kmh", "occupancy", "free_speed_kmh"]
    )

    speed_threshold = 0.60 * last["free_speed_kmh"]
    occupancy_threshold = last["occupancy"].quantile(0.90)

    last["speed_only"] = (
        last["speed_kmh"] <= speed_threshold
    )
    last["speed_and_occupancy"] = (
        last["speed_only"]
        | (last["occupancy"] >= occupancy_threshold)
    )

    results.append({
        "panel": panel,
        "eligible_rows": len(last),
        "occupancy_p90_threshold": occupancy_threshold,
        "speed_only_count": int(last["speed_only"].sum()),
        "combined_count": int(last["speed_and_occupancy"].sum()),
        "added_by_occupancy": int(
            (last["speed_and_occupancy"] & ~last["speed_only"]).sum()
        ),
        "speed_only_rate_pct": 100 * last["speed_only"].mean(),
        "combined_rate_pct": 100 * last["speed_and_occupancy"].mean(),
    })

result = pd.DataFrame(results)
print(result.round(4).to_string(index=False))

In [ ]:
RELEASE = Path(
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"
)

panels = [
    "D7_I405_S", "D7_I405_N",
    "D7_I10_E", "D7_I10_W",
    "D7_I210_W", "D7_I210_E",
    "D12_I5_N", "D12_I5_S"
]

results = []

for panel in panels:
    train_dir = RELEASE / "task2" / panel / "train"

    history = pd.read_parquet(train_dir / "window_history.parquet")
    links = pd.read_csv(
        RELEASE / "corridors" / panel / "network" / "links.csv"
    )

    history["timestamp"] = pd.to_datetime(
        history["timestamp"], utc=True
    )
    history["link_id"] = history["link_id"].astype(str)
    links["link_id"] = links["link_id"].astype(str)

    history = history.merge(
        links[["link_id", "free_speed_kmh"]],
        on="link_id",
        how="left",
        validate="many_to_one"
    )

    history = history.sort_values(
        ["window_id", "link_id", "timestamp"]
    ).copy()

    history["occupancy"] = pd.to_numeric(
        history["occupancy"], errors="coerce"
    )

    # Per-link reference threshold, estimated from training history.
    link_occ_p90 = history.groupby("link_id")["occupancy"].quantile(0.90)
    history["occ_threshold"] = history["link_id"].map(link_occ_p90)

    history["high_occ"] = (
        history["occupancy"].notna()
        & (history["occupancy"] >= history["occ_threshold"])
    ).astype(float)

    history.loc[
        history["occupancy"].isna(), "high_occ"
    ] = np.nan

    group_keys = ["window_id", "link_id"]
    grouped = history.groupby(group_keys, sort=False)["high_occ"]

    history["high_occ_share3"] = grouped.transform(
        lambda s: s.rolling(3, min_periods=1).mean()
    )
    history["high_occ_count3"] = grouped.transform(
        lambda s: s.rolling(3, min_periods=1).count()
    )

    last_ts = history.groupby("window_id")["timestamp"].transform("max")
    last = history.loc[
        history["timestamp"].eq(last_ts)
        & history["is_score_eligible"].astype(bool)
    ].copy()

    last = last.dropna(
        subset=["speed_kmh", "free_speed_kmh", "occupancy"]
    )

    last["speed_queue"] = (
        last["speed_kmh"] <= 0.60 * last["free_speed_kmh"]
    )

    # Occupancy must be high in at least two of the last three readings.
    last["persistent_high_occ"] = last["high_occ_share3"] >= (2 / 3)

    last["combined_rule"] = (
        last["speed_queue"] | last["persistent_high_occ"]
    )

    results.append({
        "panel": panel,
        "eligible_rows": len(last),
        "speed_queue_count": int(last["speed_queue"].sum()),
        "persistent_high_occ_count": int(
            last["persistent_high_occ"].sum()
        ),
        "combined_count": int(last["combined_rule"].sum()),
        "occupancy_added": int(
            (last["combined_rule"] & ~last["speed_queue"]).sum()
        ),
    })

result = pd.DataFrame(results)
print(result.to_string(index=False))